<div class="alert alert-danger" role="alert">
<h1 align="center"><font size = 14>Unsupervised PINNs for free-vibration eigenanalysis of 2-D FG Euler–Bernoulli nanobeams under NSGT (S-S)</font></h1>
<h3 align="center">Armin Amani</h3>
<h4 align="center">E-mail address: arminamani8251@gmail.com<h4>

<div class="alert alert-danger" role="alert">
📤 Import Libraries

In [ ]:
from __future__ import annotations

import argparse
import copy
import gc
import json
import logging
import math
import random
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

try:
    from scipy.integrate import cumulative_trapezoid, solve_bvp
    SCIPY_AVAILABLE = True
except Exception:
    cumulative_trapezoid = None
    solve_bvp = None
    SCIPY_AVAILABLE = False

<div class="alert alert-danger" role="alert"> 
🔎 Configuration Setting and Determine Parameters

In [ ]:
torch.set_default_dtype(torch.float64)

LOGGER = logging.getLogger("pinn_2dfg_eb_nsgt_ss")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)


# Configuration

@dataclass
class Config:
    # Material: ceramic = Al2O3, metal = steel
    E_c: float = 390.0e9
    rho_c: float = 3960.0
    E_m: float = 210.0e9
    rho_m: float = 7800.0

    # 2D-FG exponents
    k: float = 1.0
    beta: float = 1.0

    # Geometry
    h: float = 17.6e-6
    b_over_h: float = 2.0
    L_over_h: float = 30.0

    """
    Dimensionless NSGT length parameters
    
    tau  = mu/L = ea/L
    zeta = ell/L
    
    zeta must be > 0 for the present sixth-order S-S NSGT formulation.
    These are baseline study values, not universal material constants.
    """
    tau: float = 0.05
    zeta: float = 0.05

    """
    PINN collocation and quadrature
    
    PDE collocation points are interior points.
    
    For the adopted S-S formulation, the kinematic conditions
    
        Phi(0)  = Phi''(0) = 0,
        Phi(1)  = Phi''(1) = 0,
    
    are imposed exactly through the hard trial-function architecture,
    while the two bending-moment conditions
    
        Mbar(0) = 0,
        Mbar(1) = 0,
    
    remain active soft physics constraints in the BC loss.
    
    The same fixed collocation and quadrature sets are retained during
    the L-BFGS phase so that every closure evaluates a deterministic
    objective throughout the line search.
    """
    n_colloc: int = 128
    n_quad: int = 128
    n_test: int = 401

    x_min: float = 0.0
    x_max: float = 1.0

    """
    Network: X -> Phi(X)
    
    Biases remain enabled so that the hard S-S trial-function
    parametrization does not introduce unintended endpoint restrictions.
    """
    hidden_1: int = 128
    hidden_2: int = 64
    hidden_3: int = 32

    use_bias: bool = True

    """
    Hybrid optimization
    
    Phase I  : Adam for global exploration / initial convergence.
    Phase II : Full-batch L-BFGS for deterministic quasi-Newton refinement.
    
    The L-BFGS quantity below is a maximum number of quasi-Newton
    iterations, not an Adam-style epoch count.
    
    No reference eigenvalue or eigenfunction is used by either optimizer.
    """

    # Phase I: Adam
    adam_epochs: int = 20_000

    lr_nn: float = 2.0e-3
    lr_lambda: float = 1.0e-3

    lambda_init: float = 100.0

    # Absolute Adam learning-rate milestones.
    lr_milestones: Tuple[int, ...] = (
        3_000,
        4_250,
        15_000,
    )

    lr_gamma: float = 0.20

    # Phase II: L-BFGS
    lbfgs_max_iter: int = 10_000
    lbfgs_block_iter: int = 100

    lbfgs_lr: float = 1.0
    lbfgs_history_size: int = 100

    lbfgs_tolerance_grad: float = 1.0e-9
    lbfgs_tolerance_change: float = 1.0e-12

    lbfgs_line_search_fn: str = "strong_wolfe"

    """
    Physics-informed objective
    
    Four S-S kinematic endpoint conditions are hard-imposed:
    
        Phi(0)  = Phi''(0) = 0,
        Phi(1)  = Phi''(1) = 0.
    
    The two bending-moment conditions remain active soft constraints:
    
        Mbar(0) = 0,
        Mbar(1) = 0.
    
    Therefore alpha_bc must remain active for the S-S branch.
    """
    alpha_de: float = 1.0
    alpha_bc: float = 5.0

    # The eigenfunction is differentiably r-weighted normalized inside
    # compute_loss. Therefore no separate normalization penalty is required.
    alpha_norm: float = 0.0

    l2_coeff: float = 1.0e-6
    apply_l2: bool = False

    # Optional Adam gradient clipping.
    grad_clip_norm: Optional[float] = None

    # Reproducibility
    seed: Optional[int] = 42
    deterministic_torch: bool = True

    # Logging / output
    log_every: int = 50
    plot_every: int = 500
    save_epoch_plots: bool = True

    output_dir: str = (
        "results_2dfg_eb_nsgt_ss_"
        "adam20k_lbfgs10k_seed42"
    )

    device: str = "auto"

    """
    Independent SciPy BVP reference
    
    The BVP solution is used only for independent validation and optional
    visualization. It must not enter:
    
      - the PINN objective,
      - Adam updates,
      - L-BFGS updates,
      - optimizer switching,
      - checkpoint selection,
      - stopping criteria,
      - learning-rate scheduling.
    """
    run_reference_bvp: bool = True

    reference_lambda_guess: float = 100.0
    reference_tol: float = 1.0e-7
    reference_max_nodes: int = 20_000

    # Derived geometry / reference quantities
    @property
    def b(self) -> float:
        return self.b_over_h * self.h

    @property
    def L(self) -> float:
        return self.L_over_h * self.h

    @property
    def area(self) -> float:
        return self.b * self.h

    @property
    def I_g(self) -> float:
        return self.b * self.h**3 / 12.0

    @property
    def D0(self) -> float:
        return self.E_c * self.I_g

    @property
    def m0(self) -> float:
        return self.rho_c * self.area

    @property
    def mu(self) -> float:
        return self.tau * self.L

    @property
    def ell(self) -> float:
        return self.zeta * self.L

<div class="alert alert-danger" role="alert"> 
🔎 Validation & seeds for reproducibility

In [ ]:
# Validation, reproducibility, and device

def validate_config(cfg: Config) -> None:
    # Material properties
    if cfg.E_c <= 0.0 or cfg.E_m <= 0.0:
        raise ValueError(
            "Elastic moduli must be positive."
        )

    if cfg.rho_c <= 0.0 or cfg.rho_m <= 0.0:
        raise ValueError(
            "Material densities must be positive."
        )

    # Geometry
    if (
        cfg.h <= 0.0
        or cfg.b_over_h <= 0.0
        or cfg.L_over_h <= 0.0
    ):
        raise ValueError(
            "Geometry parameters h, b/h, and L/h must be positive."
        )

    # 2D-FG gradation parameters
    if cfg.k < 0.0 or cfg.beta < 0.0:
        raise ValueError(
            "The adopted power-law gradation requires "
            "k >= 0 and beta >= 0."
        )

    # NSGT parameters
    if cfg.tau < 0.0:
        raise ValueError(
            "tau must be nonnegative."
        )

    if cfg.zeta <= 0.0:
        raise ValueError(
            "zeta must be strictly positive for the present sixth-order "
            "S-S NSGT formulation with six boundary conditions. "
            "The zeta=0 limit requires a separate reduced-order formulation."
        )

    # Computational domain
    if not (
        math.isfinite(cfg.x_min)
        and math.isfinite(cfg.x_max)
    ):
        raise ValueError(
            "x_min and x_max must be finite."
        )

    if cfg.x_max <= cfg.x_min:
        raise ValueError(
            "x_max must be greater than x_min."
        )

    # PINN discretization
    if cfg.n_colloc < 16:
        raise ValueError(
            "Use at least 16 interior collocation points."
        )

    if cfg.n_quad < 16:
        raise ValueError(
            "Use at least 16 quadrature points."
        )

    if cfg.n_test < 2:
        raise ValueError(
            "n_test must be at least 2."
        )

    # Neural network
    if (
        cfg.hidden_1 < 1
        or cfg.hidden_2 < 1
        or cfg.hidden_3 < 1
    ):
        raise ValueError(
            "All hidden-layer widths must be positive integers."
        )

    if not cfg.use_bias:
        raise ValueError(
            "use_bias=True is required for the adopted hard S-S "
            "trial-function parametrization to avoid unintended "
            "additional endpoint restrictions."
        )

    # Eigenvalue initialization
    if (
        not math.isfinite(cfg.lambda_init)
        or cfg.lambda_init <= 0.0
    ):
        raise ValueError(
            "lambda_init must be finite and strictly positive."
        )

    # Phase I — Adam
    if cfg.adam_epochs < 1:
        raise ValueError(
            "adam_epochs must be positive."
        )

    if (
        not math.isfinite(cfg.lr_nn)
        or cfg.lr_nn <= 0.0
    ):
        raise ValueError(
            "lr_nn must be finite and positive."
        )

    if (
        not math.isfinite(cfg.lr_lambda)
        or cfg.lr_lambda <= 0.0
    ):
        raise ValueError(
            "lr_lambda must be finite and positive."
        )

    if (
        not math.isfinite(cfg.lr_gamma)
        or cfg.lr_gamma <= 0.0
        or cfg.lr_gamma > 1.0
    ):
        raise ValueError(
            "lr_gamma must satisfy 0 < lr_gamma <= 1."
        )

    if len(cfg.lr_milestones) == 0:
        raise ValueError(
            "At least one Adam learning-rate milestone is required."
        )

    if (
        tuple(
            sorted(
                set(cfg.lr_milestones)
            )
        )
        != cfg.lr_milestones
    ):
        raise ValueError(
            "lr_milestones must be strictly increasing and unique."
        )

    if any(
        milestone <= 0
        or milestone >= cfg.adam_epochs
        for milestone in cfg.lr_milestones
    ):
        raise ValueError(
            "Each Adam LR milestone must satisfy "
            "0 < milestone < adam_epochs."
        )

    # Phase II — L-BFGS
    if cfg.lbfgs_max_iter < 1:
        raise ValueError(
            "lbfgs_max_iter must be positive."
        )

    if cfg.lbfgs_block_iter < 1:
        raise ValueError(
            "lbfgs_block_iter must be positive."
        )

    if cfg.lbfgs_block_iter > cfg.lbfgs_max_iter:
        raise ValueError(
            "lbfgs_block_iter must not exceed lbfgs_max_iter."
        )

    if (
        not math.isfinite(cfg.lbfgs_lr)
        or cfg.lbfgs_lr <= 0.0
    ):
        raise ValueError(
            "lbfgs_lr must be finite and positive."
        )

    if cfg.lbfgs_history_size < 1:
        raise ValueError(
            "lbfgs_history_size must be positive."
        )

    if (
        not math.isfinite(
            cfg.lbfgs_tolerance_grad
        )
        or cfg.lbfgs_tolerance_grad < 0.0
    ):
        raise ValueError(
            "lbfgs_tolerance_grad must be finite and nonnegative."
        )

    if (
        not math.isfinite(
            cfg.lbfgs_tolerance_change
        )
        or cfg.lbfgs_tolerance_change < 0.0
    ):
        raise ValueError(
            "lbfgs_tolerance_change must be finite and nonnegative."
        )

    if cfg.lbfgs_line_search_fn not in (
        None,
        "strong_wolfe",
    ):
        raise ValueError(
            "lbfgs_line_search_fn must be None or 'strong_wolfe'."
        )

    # Physics-informed objective
    if (
        not math.isfinite(cfg.alpha_de)
        or cfg.alpha_de <= 0.0
    ):
        raise ValueError(
            "alpha_de must be finite and strictly positive."
        )

    # For the present S-S formulation, the two bending-moment conditions
    # are active soft constraints and therefore require a nonzero BC weight.
    if (
        not math.isfinite(cfg.alpha_bc)
        or cfg.alpha_bc <= 0.0
    ):
        raise ValueError(
            "alpha_bc must be finite and strictly positive for the present "
            "S-S formulation because the two endpoint bending-moment "
            "conditions are enforced through the BC loss."
        )

    if (
        not math.isfinite(cfg.alpha_norm)
        or cfg.alpha_norm < 0.0
    ):
        raise ValueError(
            "alpha_norm must be finite and nonnegative."
        )

    if (
        not math.isfinite(cfg.l2_coeff)
        or cfg.l2_coeff < 0.0
    ):
        raise ValueError(
            "l2_coeff must be finite and nonnegative."
        )

    if (
        cfg.grad_clip_norm is not None
        and (
            not math.isfinite(
                cfg.grad_clip_norm
            )
            or cfg.grad_clip_norm <= 0.0
        )
    ):
        raise ValueError(
            "grad_clip_norm must be None or a finite positive value."
        )

    # Logging / plotting
    if cfg.log_every < 0:
        raise ValueError(
            "log_every must be nonnegative."
        )

    if cfg.plot_every < 0:
        raise ValueError(
            "plot_every must be nonnegative."
        )

    if (
        cfg.save_epoch_plots
        and cfg.plot_every < 1
    ):
        raise ValueError(
            "plot_every must be positive when save_epoch_plots=True."
        )

    if not str(cfg.output_dir).strip():
        raise ValueError(
            "output_dir must be a nonempty path string."
        )

    # Independent BVP reference
    if (
        not math.isfinite(
            cfg.reference_lambda_guess
        )
        or cfg.reference_lambda_guess <= 0.0
    ):
        raise ValueError(
            "reference_lambda_guess must be finite and positive."
        )

    if (
        not math.isfinite(cfg.reference_tol)
        or cfg.reference_tol <= 0.0
    ):
        raise ValueError(
            "reference_tol must be finite and positive."
        )

    if cfg.reference_max_nodes < 10:
        raise ValueError(
            "reference_max_nodes must be at least 10."
        )

    # S-S endpoint regularity
    is_integer_k = (
        abs(
            cfg.k
            - round(cfg.k)
        )
        < 1.0e-12
    )

    if (
        not is_integer_k
        and cfg.k < 1.0
    ):
        raise ValueError(
            "For the present strong-form S-S formulation, noninteger k < 1 "
            "is not admissible because the endpoint bending-moment operator "
            "requires finite d'(0), whereas the derivative of X^k contains "
            "the singular factor X^(k-1) at X=0."
        )

    # Additional regularity required only by the independent expanded BVP
    if (
        not is_integer_k
        and cfg.k < 3.0
    ):
        warnings.warn(
            "Noninteger 1 <= k < 3 is admissible for the present S-S PINN, "
            "but coefficient derivatives required by the expanded-form "
            "BVP reference, notably d''' and r'', are singular at X=0. "
            "The independent BVP validation will therefore be skipped.",
            RuntimeWarning,
        )


def set_seed(
    seed: Optional[int],
    deterministic: bool = False,
) -> None:
    if seed is None:
        return

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.use_deterministic_algorithms(
            True,
            warn_only=True,
        )

        if torch.cuda.is_available():
            torch.backends.cudnn.benchmark = False
            torch.backends.cudnn.deterministic = True


def resolve_device(
    device_arg: str,
) -> torch.device:
    if device_arg == "auto":
        return torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )

    return torch.device(
        device_arg
    )

<div class="alert alert-danger" role="alert"> 
🔎 2D-FG section resultants and dimensionless coefficient functions

In [ ]:
# 2D-FG section resultants and dimensionless coefficient functions

def _section_constants(cfg: Config) -> Tuple[float, float, float]:
    """Thickness power-law integrals used by A11, B11 and D11."""
    beta = cfg.beta
    cA = 1.0 / (beta + 1.0)
    cB = beta / (2.0 * (beta + 1.0) * (beta + 2.0))
    cD = (beta**2 + beta + 2.0) / (
        4.0 * (beta + 1.0) * (beta + 2.0) * (beta + 3.0)
    )
    return cA, cB, cD


def section_resultants_numpy(X: np.ndarray, cfg: Config) -> Dict[str, np.ndarray]:
    """Closed-form A11, B11, D11, D*, I0, I1, I2 for the 2D-FG law."""
    X = np.asarray(X, dtype=np.float64)
    q = np.power(X, cfg.k)
    cA, cB, cD = _section_constants(cfg)

    dE = cfg.E_m - cfg.E_c
    drho = cfg.rho_m - cfg.rho_c

    A11 = cfg.b * cfg.h * (cfg.E_c + dE * cA * q)
    B11 = cfg.b * cfg.h**2 * (dE * cB * q)
    D11 = cfg.b * cfg.h**3 * (cfg.E_c / 12.0 + dE * cD * q)
    Dstar = D11 - B11**2 / A11

    I0 = cfg.b * cfg.h * (cfg.rho_c + drho * cA * q)
    I1 = cfg.b * cfg.h**2 * (drho * cB * q)
    I2 = cfg.b * cfg.h**3 * (cfg.rho_c / 12.0 + drho * cD * q)

    zE = B11 / A11
    zrho = I1 / I0

    d = Dstar / cfg.D0
    r = I0 / cfg.m0

    return {
        "A11": A11,
        "B11": B11,
        "D11": D11,
        "Dstar": Dstar,
        "I0": I0,
        "I1": I1,
        "I2": I2,
        "zE": zE,
        "zrho": zrho,
        "d": d,
        "r": r,
    }


def dimensionless_coefficients_torch(
    X: torch.Tensor,
    cfg: Config,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Return d(X)=D*/D0 and r(X)=I0/m0 in a scale-safe dimensionless form.

    The formula is algebraically identical to the physical resultants above,
    but avoids carrying E~1e11 and h~1e-5 through high-order AD.
    """
    q = X.pow(cfg.k)
    cA, cB, cD = _section_constants(cfg)

    eta_E = (cfg.E_m - cfg.E_c) / cfg.E_c
    eta_rho = (cfg.rho_m - cfg.rho_c) / cfg.rho_c

    # A11/(b h Ec), B11/(b h^2 Ec), D11/(b h^3 Ec)
    Abar = 1.0 + eta_E * cA * q
    Bbar = eta_E * cB * q
    Dbar = 1.0 / 12.0 + eta_E * cD * q

    # D0 = Ec b h^3 / 12
    d = 12.0 * (Dbar - Bbar.square() / Abar)
    r = 1.0 + eta_rho * cA * q
    return d, r


def verify_closed_form_section_resultants(cfg: Config, n_z: int = 120) -> None:
    """Numerically integrate through thickness and verify all closed forms."""
    xi, w = np.polynomial.legendre.leggauss(n_z)
    xi = 0.5 * (xi + 1.0)
    w = 0.5 * w
    z = cfg.h * (xi - 0.5)

    Xs = np.array([0.0, 0.2, 0.5, 0.8, 1.0], dtype=np.float64)
    exact = section_resultants_numpy(Xs, cfg)

    num = {key: [] for key in ["A11", "B11", "D11", "I0", "I1", "I2"]}
    for X in Xs:
        qx = X**cfg.k
        E = cfg.E_c + (cfg.E_m - cfg.E_c) * qx * xi**cfg.beta
        rho = cfg.rho_c + (cfg.rho_m - cfg.rho_c) * qx * xi**cfg.beta
        dA_weight = cfg.b * cfg.h * w

        num["A11"].append(np.sum(E * dA_weight))
        num["B11"].append(np.sum(z * E * dA_weight))
        num["D11"].append(np.sum(z**2 * E * dA_weight))
        num["I0"].append(np.sum(rho * dA_weight))
        num["I1"].append(np.sum(z * rho * dA_weight))
        num["I2"].append(np.sum(z**2 * rho * dA_weight))

    max_rel = 0.0
    for key in num:
        a = np.asarray(num[key])
        b = np.asarray(exact[key])
        # Use a global characteristic scale for each resultant.  B11 and I1
        # are exactly zero at X=0 for k>0, so pointwise relative error would
        # incorrectly amplify harmless Gauss-quadrature cancellation noise.
        scale = max(float(np.max(np.abs(b))), 1.0e-300)
        rel = float(np.max(np.abs(a - b)) / scale)
        max_rel = max(max_rel, rel)

    LOGGER.info("Closed-form section-resultant check: max relative error = %.3e", max_rel)
    if max_rel > 1.0e-9:
        raise RuntimeError("Section-resultant closed-form verification failed.")


def validate_coefficient_profiles(cfg: Config) -> None:
    X = np.linspace(0.0, 1.0, 1001)
    sec = section_resultants_numpy(X, cfg)
    if not np.all(np.isfinite(sec["d"])) or not np.all(np.isfinite(sec["r"])):
        raise ValueError("Non-finite d(X) or r(X) detected.")
    if np.min(sec["d"]) <= 0:
        raise ValueError("Condensed bending stiffness D*(X) is non-positive.")
    if np.min(sec["r"]) <= 0:
        raise ValueError("Mass coefficient r(X) is non-positive.")

    LOGGER.info(
        "Coefficient ranges: d(X)=[%.6g, %.6g], r(X)=[%.6g, %.6g]",
        np.min(sec["d"]), np.max(sec["d"]), np.min(sec["r"]), np.max(sec["r"]),
    )
    LOGGER.info(
        "Max |z_E|/h = %.6g, max |z_rho|/h = %.6g; I1^rho is %s in general.",
        np.max(np.abs(sec["zE"])) / cfg.h,
        np.max(np.abs(sec["zrho"])) / cfg.h,
        "nonzero" if np.max(np.abs(sec["I1"])) > 1.0e-30 else "zero",
    )

<div class="alert alert-danger" role="alert"> 
🔎 NeuralNetwork

In [ ]:
class NeuralNetwork(nn.Module):
    """X -> Phi(X), 1 -> 128 -> 64 -> 32 -> 1 by default, SiLU/Swish."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.fc1 = nn.Linear(1, cfg.hidden_1, bias=cfg.use_bias)
        self.fc2 = nn.Linear(cfg.hidden_1, cfg.hidden_2, bias=cfg.use_bias)
        self.fc3 = nn.Linear(cfg.hidden_2, cfg.hidden_3, bias=cfg.use_bias)
        self.out = nn.Linear(cfg.hidden_3, 1, bias=cfg.use_bias)
        self.act = nn.SiLU()
        self._glorot_uniform_init()

    def _glorot_uniform_init(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        X = self.act(self.fc1(X))
        X = self.act(self.fc2(X))
        X = self.act(self.fc3(X))
        return self.out(X)

<div class="alert alert-danger" role="alert"> 
🔎 EigenPINN

In [ ]:
class EigenPINN(nn.Module):
    """
    Hard-constrained eigenvalue PINN for the present Li-type
    simply-supported (S-S) fixed-curvature formulation.

    The adopted S-S boundary conditions are

        Phi(0)   = 0,
        Phi''(0) = 0,
        M_bar(0) = 0,

        Phi(1)   = 0,
        Phi''(1) = 0,
        M_bar(1) = 0.

    The trial-function architecture imposes exactly the four kinematic /
    fixed-curvature conditions

        Phi(0)   = Phi''(0) = 0,
        Phi(1)   = Phi''(1) = 0,

    while leaving the two endpoint slopes free.

    The bending-moment conditions are NOT embedded in the trial function;
    they remain active physics-informed boundary constraints and are
    enforced through the BC loss.

    -------------------------------------------------------------------------
    Complete hard-constrained S-S representation
    -------------------------------------------------------------------------

    Let

        xi = (X - x_min) / (x_max - x_min),

    and define two quintic Hermite-type boundary functions

        H_L(xi)
            = xi - 6 xi^3 + 8 xi^4 - 3 xi^5,

        H_R(xi)
            = -4 xi^3 + 7 xi^4 - 3 xi^5.

    They satisfy

        H_L(0)   = H_L''(0) = 0,
        H_L(1)   = H_L''(1) = 0,
        H_L'(0)  = 1,
        H_L'(1)  = 0,

    and

        H_R(0)   = H_R''(0) = 0,
        H_R(1)   = H_R''(1) = 0,
        H_R'(0)  = 0,
        H_R'(1)  = 1.

    The interior residual envelope is

        B(xi) = 64 xi^3 (1-xi)^3,

    for which

        B = B' = B'' = 0

    at both endpoints.

    The raw eigenfunction is represented as

        Phi_raw(X)
            = L * s_L * H_L(xi)
            + L * s_R * H_R(xi)
            + B(xi) N_theta(X),

    where

        L = x_max - x_min,

    and s_L and s_R are independent trainable endpoint slopes.

    Consequently,

        Phi_raw(0)   = Phi_raw''(0) = 0,
        Phi_raw(1)   = Phi_raw''(1) = 0

    identically for every set of trainable parameters, while

        Phi_raw'(0) = s_L,
        Phi_raw'(1) = s_R

    remain free.

    Unlike the previous multiplicative coordinate-map construction, this
    representation does not impose hidden algebraic relations among higher
    endpoint derivatives. Any sufficiently smooth function satisfying only
    the four hard S-S kinematic conditions can be represented by suitable
    endpoint slopes and a suitable latent interior function.

    The positive eigenvalue is parameterized logarithmically as

        lambda = exp(lambda_raw) > 0.
    """

    def __init__(self, cfg: Config):
        super().__init__()

        if not cfg.use_bias:
            raise ValueError(
                "The adopted hard S-S parametrization requires "
                "use_bias=True to preserve a sufficiently general latent "
                "interior approximation space."
            )

        if (
            not math.isfinite(cfg.lambda_init)
            or cfg.lambda_init <= 0.0
        ):
            raise ValueError(
                "lambda_init must be finite and strictly positive."
            )

        self.nn_phi = NeuralNetwork(cfg)

        self.x_min = float(
            cfg.x_min
        )

        self.x_max = float(
            cfg.x_max
        )

        if self.x_max <= self.x_min:
            raise ValueError(
                "Require x_max > x_min."
            )

        self.domain_length = (
            self.x_max
            - self.x_min
        )

        # Independent trainable endpoint slopes
        self.slope_left = nn.Parameter(
            torch.tensor(
                1.0,
                dtype=torch.float64,
            )
        )

        self.slope_right = nn.Parameter(
            torch.tensor(
                -1.0,
                dtype=torch.float64,
            )
        )

        # Positive trainable eigenvalue
        self.lambda_raw = nn.Parameter(
            torch.tensor(
                math.log(cfg.lambda_init),
                dtype=torch.float64,
            )
        )

    @property
    def lambda_(self) -> torch.Tensor:
        return torch.exp(
            self.lambda_raw
        )

    def normalized_coordinate(
        self,
        X: torch.Tensor,
    ) -> torch.Tensor:
        """
        Map X in [x_min, x_max] to xi in [0,1].
        """
        return (
            (X - self.x_min)
            / self.domain_length
        )

    def ss_left_slope_basis(
        self,
        X: torch.Tensor,
    ) -> torch.Tensor:
        """
        Quintic left-slope basis

            H_L(xi)
                = xi
                - 6 xi^3
                + 8 xi^4
                - 3 xi^5.

        It satisfies

            H_L(0)   = H_L''(0) = 0,
            H_L(1)   = H_L''(1) = 0,

            H_L'(0)  = 1,
            H_L'(1)  = 0.
        """
        xi = self.normalized_coordinate(
            X
        )

        return (
            xi
            - 6.0 * xi.pow(3)
            + 8.0 * xi.pow(4)
            - 3.0 * xi.pow(5)
        )

    def ss_right_slope_basis(
        self,
        X: torch.Tensor,
    ) -> torch.Tensor:
        """
        Quintic right-slope basis

            H_R(xi)
                = -4 xi^3
                + 7 xi^4
                - 3 xi^5.

        It satisfies

            H_R(0)   = H_R''(0) = 0,
            H_R(1)   = H_R''(1) = 0,

            H_R'(0)  = 0,
            H_R'(1)  = 1.
        """
        xi = self.normalized_coordinate(
            X
        )

        return (
            -4.0 * xi.pow(3)
            + 7.0 * xi.pow(4)
            - 3.0 * xi.pow(5)
        )

    def ss_interior_factor(
        self,
        X: torch.Tensor,
    ) -> torch.Tensor:
        """
        Interior hard-constraint envelope

            B(xi) = 64 xi^3 (1-xi)^3.

        It satisfies

            B = B' = B'' = 0

        at both endpoints.

        The factor 64 gives B(1/2)=1 and only improves numerical scaling.
        """
        xi = self.normalized_coordinate(
            X
        )

        return (
            64.0
            * xi.pow(3)
            * (1.0 - xi).pow(3)
        )

    def forward(
        self,
        X: torch.Tensor,
    ) -> torch.Tensor:
        latent_mode = self.nn_phi(
            X
        )

        left_component = (
            self.domain_length
            * self.slope_left
            * self.ss_left_slope_basis(X)
        )

        right_component = (
            self.domain_length
            * self.slope_right
            * self.ss_right_slope_basis(X)
        )

        interior_component = (
            self.ss_interior_factor(X)
            * latent_mode
        )

        return (
            left_component
            + right_component
            + interior_component
        )

<div class="alert alert-danger" role="alert"> 
🔎 Automatic differentiation and operators

In [ ]:
# Automatic differentiation and operators

def derivative(y: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    return torch.autograd.grad(
        outputs=y,
        inputs=x,
        grad_outputs=torch.ones_like(y),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]


def nth_derivative(y: torch.Tensor, x: torch.Tensor, order: int) -> torch.Tensor:
    out = y
    for _ in range(order):
        out = derivative(out, x)
    return out


def phi_derivatives(
    model: EigenPINN,
    X: torch.Tensor,
    max_order: int,
) -> Tuple[torch.Tensor, ...]:
    if not X.requires_grad:
        raise ValueError("X must have requires_grad=True for spatial differentiation.")
    values = [model(X)]
    for _ in range(max_order):
        values.append(derivative(values[-1], X))
    return tuple(values)


def compact_governing_residual(
    model: EigenPINN,
    X: torch.Tensor,
    cfg: Config,
) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
    """
    Compact operator implementation of
      R = (d Phi'')'' - zeta^2 (d Phi''')'''
          - lambda [r Phi - tau^2 (r Phi)''].

    This form is preferred for training because it mirrors the derived
    divergence/operator structure and avoids manually coding d', d'', d'''.
    """
    if not X.requires_grad:
        raise ValueError("X must require gradients.")

    phi = model(X)
    phi1 = derivative(phi, X)
    phi2 = derivative(phi1, X)
    phi3 = derivative(phi2, X)

    d, r = dimensionless_coefficients_torch(X, cfg)

    bending_flux = d * phi2
    bending = derivative(derivative(bending_flux, X), X)

    gradient_flux = d * phi3
    gradient = derivative(derivative(derivative(gradient_flux, X), X), X)

    mass_product = r * phi
    mass_second = derivative(derivative(mass_product, X), X)
    mass_operator = r * phi - (cfg.tau**2) * mass_second

    residual = bending - (cfg.zeta**2) * gradient - model.lambda_ * mass_operator

    return residual, {
        "phi": phi,
        "phi1": phi1,
        "phi2": phi2,
        "phi3": phi3,
        "d": d,
        "r": r,
        "bending_operator": bending,
        "gradient_operator": gradient,
        "mass_operator": mass_operator,
    }


def verify_compact_vs_expanded_operator(cfg: Config, device: torch.device) -> None:
    """Algebraic/AD self-check of compact and expanded governing operators."""
    X = torch.linspace(0.08, 0.92, 31, device=device).reshape(-1, 1)
    X.requires_grad_(True)

    # Smooth nontrivial test function independent of the PINN.
    phi = X**3 * (1.0 - X)**3 * torch.exp(0.3 * X)
    phi1 = derivative(phi, X)
    phi2 = derivative(phi1, X)
    phi3 = derivative(phi2, X)
    phi4 = derivative(phi3, X)
    phi5 = derivative(phi4, X)
    phi6 = derivative(phi5, X)

    d, r = dimensionless_coefficients_torch(X, cfg)
    d1 = derivative(d, X)
    d2 = derivative(d1, X)
    d3 = derivative(d2, X)
    r1 = derivative(r, X)
    r2 = derivative(r1, X)

    lam = torch.tensor(123.456, dtype=X.dtype, device=device)

    compact = (
        derivative(derivative(d * phi2, X), X)
        - cfg.zeta**2 * derivative(derivative(derivative(d * phi3, X), X), X)
        - lam * (r * phi - cfg.tau**2 * derivative(derivative(r * phi, X), X))
    )

    expanded = (
        d * phi4 + 2.0 * d1 * phi3 + d2 * phi2
        - cfg.zeta**2
        * (d * phi6 + 3.0 * d1 * phi5 + 3.0 * d2 * phi4 + d3 * phi3)
        - lam
        * (
            r * phi
            - cfg.tau**2 * (r * phi2 + 2.0 * r1 * phi1 + r2 * phi)
        )
    )

    abs_err = torch.max(torch.abs(compact - expanded)).item()
    scale = max(torch.max(torch.abs(expanded)).item(), 1.0)
    rel_err = abs_err / scale
    LOGGER.info(
        "Compact-vs-expanded operator check: max abs=%.3e, relative=%.3e",
        abs_err,
        rel_err,
    )
    if rel_err > 5.0e-9:
        raise RuntimeError("Compact and expanded operator forms are inconsistent.")

<div class="alert alert-danger" role="alert"> 
🔎 Quadrature, boundary conditions, loss

In [ ]:
# Quadrature, S-S boundary conditions, regularization, and PINN loss

def gauss_legendre_tensors(
    n: int,
    device: torch.device,
    x_min: float = 0.0,
    x_max: float = 1.0,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Return Gauss-Legendre quadrature nodes and weights over [x_min, x_max].
    """
    if n < 1:
        raise ValueError(
            "Number of quadrature points must be positive."
        )

    if x_max <= x_min:
        raise ValueError(
            "Quadrature interval must satisfy x_max > x_min."
        )

    xi, wi = np.polynomial.legendre.leggauss(
        n
    )

    X = (
        0.5 * (x_max - x_min) * xi
        + 0.5 * (x_max + x_min)
    )

    W = (
        0.5
        * (x_max - x_min)
        * wi
    )

    X_tensor = torch.tensor(
        X,
        dtype=torch.float64,
        device=device,
    ).reshape(-1, 1)

    W_tensor = torch.tensor(
        W,
        dtype=torch.float64,
        device=device,
    ).reshape(-1, 1)

    return X_tensor, W_tensor


def endpoint_bc_residuals(
    model: EigenPINN,
    cfg: Config,
    device: torch.device,
) -> Dict[str, torch.Tensor]:
    """
    Evaluate endpoint quantities for the adopted Li-type S-S formulation.

    The six S-S boundary conditions are

        Phi(0)   = 0,
        Phi''(0) = 0,
        M_bar(0) = 0,

        Phi(1)   = 0,
        Phi''(1) = 0,
        M_bar(1) = 0,

    with

        M_bar
            = -d Phi''
              + zeta^2 (d Phi''')'
              - tau^2 lambda r Phi.

    The hard trial-function architecture imposes

        Phi = 0,
        Phi'' = 0

    exactly at both endpoints. Therefore, at either endpoint,

        M_bar
            = zeta^2 (d Phi''')'.

    Since zeta > 0, the physical moment condition M_bar=0 is equivalent to

        (d Phi''')' = 0.

    The rescaled quantity

        moment_core = (d Phi''')'

    is therefore used as the active S-S boundary residual. This avoids the
    small multiplicative factor zeta^2 in the optimized boundary residual
    while preserving exactly the same zero set.

    The complete dimensionless bending moment is retained independently as
    a physical diagnostic.

    Phi' is also retained only as a diagnostic; it is not constrained.
    """

    def evaluate_endpoint(
        X_value: float,
    ) -> Dict[str, torch.Tensor]:
        X = torch.tensor(
            [[X_value]],
            dtype=torch.float64,
            device=device,
            requires_grad=True,
        )

        (
            phi,
            phi1,
            phi2,
            phi3,
        ) = phi_derivatives(
            model,
            X,
            3,
        )

        d, r = dimensionless_coefficients_torch(
            X,
            cfg,
        )

        """
        Higher-order contribution to the dimensionless bending moment:
        
            (d Phi''')'
              = d' Phi''' + d Phi''''.
        """
        gradient_flux = (
            d
            * phi3
        )

        moment_core = derivative(
            gradient_flux,
            X,
        )

        # Complete physical dimensionless bending moment
        moment = (
            -d * phi2
            + (cfg.zeta**2) * moment_core
            - (cfg.tau**2)
            * model.lambda_
            * r
            * phi
        )

        return {
            "phi": phi[0, 0],
            "dphi": phi1[0, 0],
            "ddphi": phi2[0, 0],
            "moment_core": moment_core[0, 0],
            "moment": moment[0, 0],
        }

    left = evaluate_endpoint(
        cfg.x_min
    )

    right = evaluate_endpoint(
        cfg.x_max
    )

    return {
        # Left endpoint
        "phi_0": left["phi"],
        "dphi_0": left["dphi"],
        "ddphi_0": left["ddphi"],
        "moment_core_0": left["moment_core"],
        "moment_0": left["moment"],

        # Right endpoint
        "phi_1": right["phi"],
        "dphi_1": right["dphi"],
        "ddphi_1": right["ddphi"],
        "moment_core_1": right["moment_core"],
        "moment_1": right["moment"],
    }


def verify_ss_hard_constraint_structure(
    model: EigenPINN,
    cfg: Config,
    device: torch.device,
) -> None:
    """
    Verify the adopted complete hard-constrained S-S trial architecture.

    Required exact hard constraints are

        Phi(0)   = Phi''(0) = 0,
        Phi(1)   = Phi''(1) = 0.

    The endpoint rotations must remain independent:

        Phi'(0) = slope_left,
        Phi'(1) = slope_right.

    The quintic boundary bases supply the two independent endpoint slopes,
    while the interior factor satisfies

        B = B' = B'' = 0

    at both supports.

    Hence the neural-network interior correction cannot alter the four hard
    constraints or the two independently parameterized endpoint slopes.
    """

    if not model.slope_left.requires_grad:
        raise RuntimeError(
            "slope_left must be trainable."
        )

    if not model.slope_right.requires_grad:
        raise RuntimeError(
            "slope_right must be trainable."
        )

    tol_hard = 1.0e-10
    tol_slope = 1.0e-10

    endpoint_data = (
        (
            cfg.x_min,
            "left",
            model.slope_left,
        ),
        (
            cfg.x_max,
            "right",
            model.slope_right,
        ),
    )

    for (
        X_value,
        endpoint_name,
        expected_slope,
    ) in endpoint_data:
        X = torch.tensor(
            [[X_value]],
            dtype=torch.float64,
            device=device,
            requires_grad=True,
        )

        # Quintic left-slope basis
        H_left = (
            model.ss_left_slope_basis(
                X
            )
        )

        H_left_1 = derivative(
            H_left,
            X,
        )

        H_left_2 = derivative(
            H_left_1,
            X,
        )

        # Quintic right-slope basis
        H_right = (
            model.ss_right_slope_basis(
                X
            )
        )

        H_right_1 = derivative(
            H_right,
            X,
        )

        H_right_2 = derivative(
            H_right_1,
            X,
        )

        # Interior residual envelope
        B = model.ss_interior_factor(
            X
        )

        B1 = derivative(
            B,
            X,
        )

        B2 = derivative(
            B1,
            X,
        )

        # Actual trial eigenfunction
        (
            phi,
            phi1,
            phi2,
        ) = phi_derivatives(
            model,
            X,
            2,
        )

        # Hard S-S displacement / curvature constraints
        if (
            abs(
                float(
                    phi.detach().cpu()
                )
            )
            > tol_hard
        ):
            raise RuntimeError(
                "S-S hard constraint failed: "
                f"Phi is not zero at the {endpoint_name} endpoint."
            )

        if (
            abs(
                float(
                    phi2.detach().cpu()
                )
            )
            > tol_hard
        ):
            raise RuntimeError(
                "S-S hard constraint failed: "
                f"Phi'' is not zero at the {endpoint_name} endpoint."
            )

        # Interior factor must be invisible through second order at supports
        for (
            quantity_name,
            quantity,
        ) in (
            ("B", B),
            ("B'", B1),
            ("B''", B2),
        ):
            if (
                abs(
                    float(
                        quantity.detach().cpu()
                    )
                )
                > tol_hard
            ):
                raise RuntimeError(
                    "S-S interior-factor verification failed: "
                    f"{quantity_name} must vanish at the "
                    f"{endpoint_name} endpoint."
                )

        # Boundary bases themselves must have zero value and curvature
        for (
            basis_name,
            basis_value,
            basis_second,
        ) in (
            (
                "H_L",
                H_left,
                H_left_2,
            ),
            (
                "H_R",
                H_right,
                H_right_2,
            ),
        ):
            if (
                abs(
                    float(
                        basis_value.detach().cpu()
                    )
                )
                > tol_hard
            ):
                raise RuntimeError(
                    "S-S boundary-basis verification failed: "
                    f"{basis_name} must vanish at the "
                    f"{endpoint_name} endpoint."
                )

            if (
                abs(
                    float(
                        basis_second.detach().cpu()
                    )
                )
                > tol_hard
            ):
                raise RuntimeError(
                    "S-S boundary-basis verification failed: "
                    f"{basis_name}'' must vanish at the "
                    f"{endpoint_name} endpoint."
                )

        # Verify the independent endpoint-slope interpolation property.
        scaled_H_left_1 = (
            model.domain_length
            * H_left_1
        )

        scaled_H_right_1 = (
            model.domain_length
            * H_right_1
        )

        if endpoint_name == "left":
            expected_left_basis_slope = 1.0
            expected_right_basis_slope = 0.0
        else:
            expected_left_basis_slope = 0.0
            expected_right_basis_slope = 1.0

        if (
            abs(
                float(
                    scaled_H_left_1
                    .detach()
                    .cpu()
                )
                - expected_left_basis_slope
            )
            > tol_slope
        ):
            raise RuntimeError(
                "S-S left-slope basis verification failed."
            )

        if (
            abs(
                float(
                    scaled_H_right_1
                    .detach()
                    .cpu()
                )
                - expected_right_basis_slope
            )
            > tol_slope
        ):
            raise RuntimeError(
                "S-S right-slope basis verification failed."
            )

        # Actual endpoint rotation must equal the corresponding independent
        # trainable slope parameter.
        actual_slope = float(
            phi1
            .detach()
            .cpu()
        )

        target_slope = float(
            expected_slope
            .detach()
            .cpu()
        )

        if (
            abs(
                actual_slope
                - target_slope
            )
            > tol_slope
        ):
            raise RuntimeError(
                "S-S endpoint-slope verification failed: "
                f"Phi'({endpoint_name}) does not equal its independent "
                "trainable slope parameter."
            )

    LOGGER.info(
        "S-S hard-constraint verification passed: "
        "Phi=Phi''=0 exactly at both endpoints; "
        "Phi'(0) and Phi'(1) remain independent trainable rotations."
    )


def explicit_l2_penalty(
    model: EigenPINN,
) -> torch.Tensor:
    """
    Explicit L2 penalty on eigenfunction-shape trainable parameters only.

    Included:
        - neural-network weights and biases,
        - slope_left,
        - slope_right.

    Excluded:
        - lambda_raw.

    The eigenvalue is intentionally not regularized.
    """

    penalty = (
        model.lambda_raw
        .new_zeros(())
    )

    for parameter in model.nn_phi.parameters():
        penalty = (
            penalty
            + torch.sum(
                parameter.square()
            )
        )

    penalty = (
        penalty
        + model.slope_left.square()
        + model.slope_right.square()
    )

    return penalty


def compute_loss(
    model: EigenPINN,
    X_colloc_base: torch.Tensor,
    X_quad: torch.Tensor,
    W_quad: torch.Tensor,
    cfg: Config,
    device: torch.device,
) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
    """
    Physics-informed objective for the adopted Li-type S-S eigenvalue PINN.

    Four endpoint conditions are imposed exactly by the hard trial
    architecture:

        Phi(0)   = Phi''(0) = 0,
        Phi(1)   = Phi''(1) = 0.

    The remaining two physical boundary conditions are

        M_bar(0) = 0,
        M_bar(1) = 0.

    Since the four hard conditions hold exactly,

        M_bar
            = zeta^2 (d Phi''')'

    at either endpoint.

    Because zeta > 0, the active boundary residual can equivalently be
    written as

        moment_core = (d Phi''')' = 0.

    The raw mode is differentiably r-weighted normalized according to

        Phi_hat(X)
            = Phi_raw(X)
              / sqrt(
                    integral r(X) Phi_raw(X)^2 dX
                ).

    This prevents the trivial zero-amplitude eigenfunction without using
    any reference eigenvalue or eigenfunction.
    """

    # Fresh interior collocation leaf
    X = (
        X_colloc_base
        .detach()
        .clone()
        .requires_grad_(True)
    )

    # Raw governing-equation residual
    (
        residual_raw,
        fields,
    ) = compact_governing_residual(
        model=model,
        X=X,
        cfg=cfg,
    )

    # Raw r-weighted mode norm
    phi_q_raw = model(
        X_quad
    )

    _, r_q = dimensionless_coefficients_torch(
        X_quad,
        cfg,
    )

    raw_mass_norm_integral = torch.sum(
        W_quad
        * r_q
        * phi_q_raw.square()
    )

    raw_norm_value = float(
        raw_mass_norm_integral
        .detach()
        .cpu()
    )

    if (
        not math.isfinite(
            raw_norm_value
        )
        or raw_norm_value <= 1.0e-24
    ):
        raise FloatingPointError(
            "Raw eigenfunction r-weighted norm became numerically "
            "degenerate. Inspect trial-function initialization and "
            "optimization."
        )

    # IMPORTANT:
    # mode_scale must remain attached to the computational graph.
    mode_scale = torch.rsqrt(
        raw_mass_norm_integral
    )

    # Governing residual of the differentiably normalized mode
    residual = (
        mode_scale
        * residual_raw
    )

    # Scale-aware PDE diagnostics
    bending_term = (
        mode_scale
        * fields[
            "bending_operator"
        ]
    )

    gradient_term = (
        mode_scale
        * (cfg.zeta**2)
        * fields[
            "gradient_operator"
        ]
    )

    mass_term = (
        mode_scale
        * model.lambda_
        * fields[
            "mass_operator"
        ]
    )

    loss_de_raw = torch.mean(
        residual.square()
    )

    residual_rms = torch.sqrt(
        loss_de_raw
    )

    operator_scale_rms = (
        torch.sqrt(
            torch.mean(
                bending_term.square()
            )
        )
        + torch.sqrt(
            torch.mean(
                gradient_term.square()
            )
        )
        + torch.sqrt(
            torch.mean(
                mass_term.square()
            )
        )
    )

    relative_residual_rms = (
        residual_rms
        / (
            operator_scale_rms
            + torch.finfo(
                residual.dtype
            ).eps
        )
    )

    # S-S endpoint quantities for the raw mode
    bc_raw = endpoint_bc_residuals(
        model=model,
        cfg=cfg,
        device=device,
    )

    # All endpoint quantities below are expressed for the differentiably
    # normalized mode. Every quantity in endpoint_bc_residuals is linear
    # and homogeneous in Phi, so the same scalar normalization applies.
    bc = {
        key: (
            mode_scale
            * value
        )
        for (
            key,
            value,
        )
        in bc_raw.items()
    }

    # Active S-S boundary residual
    moment_bc_vector = torch.stack(
        (
            bc[
                "moment_core_0"
            ],
            bc[
                "moment_core_1"
            ],
        )
    )

    loss_bc_raw = torch.mean(
        moment_bc_vector.square()
    )

    moment_core_rms = torch.sqrt(
        loss_bc_raw
    )

    # Complete physical dimensionless moments
    physical_moment_vector = torch.stack(
        (
            bc[
                "moment_0"
            ],
            bc[
                "moment_1"
            ],
        )
    )

    physical_moment_rms = torch.sqrt(
        torch.mean(
            physical_moment_vector.square()
        )
    )

    # Diagnostics for the exactly imposed hard S-S conditions
    hard_bc_vector = torch.stack(
        (
            bc[
                "phi_0"
            ],
            bc[
                "ddphi_0"
            ],
            bc[
                "phi_1"
            ],
            bc[
                "ddphi_1"
            ],
        )
    )

    hard_bc_max_abs = torch.max(
        torch.abs(
            hard_bc_vector
        )
    )

    # Exact differentiable r-weighted normalization
    norm_integral = (
        raw_mass_norm_integral
        * mode_scale.square()
    )

    norm_error = (
        norm_integral
        - 1.0
    )

    norm_physical_raw = (
        norm_error.square()
    )

    # Retained for history/API compatibility.
    loss_norm_raw = (
        norm_physical_raw
    )

    # =========================================================================
    # Weighted physics-informed objective
    # =========================================================================
    loss_de = (
        cfg.alpha_de
        * loss_de_raw
    )

    loss_bc = (
        cfg.alpha_bc
        * loss_bc_raw
    )

    loss_norm = (
        cfg.alpha_norm
        * loss_norm_raw
    )

    # Optional L2 regularization
    if (
        cfg.apply_l2
        and cfg.l2_coeff > 0.0
    ):
        loss_l2 = (
            cfg.l2_coeff
            * explicit_l2_penalty(
                model
            )
        )
    else:
        loss_l2 = (
            model.lambda_raw
            .new_zeros(())
        )

    """
    Total S-S objective
    
    Unlike the C-C case, the S-S branch retains an active soft boundary
    contribution because the two moment conditions are not embedded in
    the trial architecture.
    
        L_total
            = L_DE
              + L_BC
              + L_Norm
              + L_L2.
    
    With alpha_norm=0 and apply_l2=False:
    
        L_total = L_DE + L_BC.
    """
    total = (
        loss_de
        + loss_bc
        + loss_norm
        + loss_l2
    )

    # Diagnostics
    comp: Dict[
        str,
        torch.Tensor,
    ] = {
        "total": total,

        "de": loss_de,
        "de_raw": loss_de_raw,

        "bc": loss_bc,
        "bc_raw": loss_bc_raw,

        "norm": loss_norm,
        "norm_raw": loss_norm_raw,
        "norm_physical_raw": norm_physical_raw,

        "l2": loss_l2,

        "lambda": model.lambda_,

        # Exact normalized r-weighted integral.
        "mass_norm_integral": norm_integral,

        # Raw mode-amplitude diagnostic.
        "raw_mass_norm_integral": raw_mass_norm_integral,

        # PDE diagnostics.
        "residual_rms": residual_rms,
        "operator_scale_rms": operator_scale_rms,
        "relative_residual_rms": relative_residual_rms,

        "max_abs_residual": torch.max(
            torch.abs(
                residual
            )
        ),

        # S-S boundary diagnostics.
        "moment_core_rms": moment_core_rms,
        "physical_moment_rms": physical_moment_rms,
        "hard_bc_max_abs": hard_bc_max_abs,
    }

    # Store all normalized endpoint quantities.
    for (
        key,
        value,
    ) in bc.items():
        comp[
            key
        ] = value

    # Additional normalized-mode diagnostics
    comp[
        "phi_rms"
    ] = (
        mode_scale
        * torch.sqrt(
            torch.mean(
                fields[
                    "phi"
                ].square()
            )
        )
    )

    comp[
        "mass_operator_rms"
    ] = (
        mode_scale
        * torch.sqrt(
            torch.mean(
                fields[
                    "mass_operator"
                ].square()
            )
        )
    )

    return total, comp

<div class="alert alert-danger" role="alert"> 
🔎 L2 penalty (Regularization)

In [ ]:
# L2 implementation consistency check

_l2_check_model = EigenPINN(
    Config()
)

required_l2_attributes = (
    "nn_phi",
    "slope_left",
    "slope_right",
    "lambda_raw",
)

missing_l2_attributes = tuple(
    name
    for name in required_l2_attributes
    if not hasattr(
        _l2_check_model,
        name,
    )
)

if missing_l2_attributes:
    raise RuntimeError(
        "EigenPINN is missing trainable attributes required by the "
        "S-S regularization implementation: "
        + ", ".join(
            missing_l2_attributes
        )
    )

# The adopted L2 term regularizes the eigenfunction-shape parameters
# (network + independent endpoint slopes), but not lambda_raw.
_l2_value = explicit_l2_penalty(
    _l2_check_model
)

if (
    _l2_value.ndim != 0
    or not torch.isfinite(
        _l2_value
    ).item()
):
    raise RuntimeError(
        "explicit_l2_penalty must return one finite scalar tensor."
    )

del _l2_check_model
del _l2_value

print("L2 implementation dependency check: OK")

L2 implementation dependency check: OK


<div class="alert alert-danger" role="alert"> 
🔎 PINN's Losses

In [ ]:
# PINN loss API consistency check

import inspect

expected_parameters = (
    "model",
    "X_colloc_base",
    "X_quad",
    "W_quad",
    "cfg",
    "device",
)

actual_parameters = tuple(
    inspect.signature(compute_loss).parameters.keys()
)

if actual_parameters != expected_parameters:
    raise RuntimeError(
        "compute_loss API mismatch.\n"
        f"Expected: {expected_parameters}\n"
        f"Found   : {actual_parameters}"
    )


# Required output components for the S-S PINN

required_history_components = {
    # Total and PDE losses
    "total",
    "de",
    "de_raw",

    # Active S-S moment-boundary loss
    "bc",
    "bc_raw",

    # Exact differentiable normalization diagnostics
    "norm",
    "norm_raw",
    "norm_physical_raw",

    # Optional regularization
    "l2",

    # Trainable eigenvalue
    "lambda",

    # Mass-normalization diagnostics
    "mass_norm_integral",
    "raw_mass_norm_integral",

    # PDE residual diagnostics
    "residual_rms",
    "operator_scale_rms",
    "relative_residual_rms",
    "max_abs_residual",

    # Left S-S endpoint
    # Phi = 0 and Phi'' = 0 are hard constraints.
    # Phi' is free.
    # moment_core = (d Phi''')' is the active moment residual.
    # moment is the complete physical dimensionless bending moment.
    "phi_0",
    "dphi_0",
    "ddphi_0",
    "moment_core_0",
    "moment_0",

    # Right S-S endpoint
    "phi_1",
    "dphi_1",
    "ddphi_1",
    "moment_core_1",
    "moment_1",

    # Aggregate S-S BC diagnostics
    "moment_core_rms",
    "physical_moment_rms",
    "hard_bc_max_abs",

    # Additional normalized-mode diagnostics
    "phi_rms",
    "mass_operator_rms",
}


# Execute a lightweight real call to compute_loss and verify its output API.

_rng_state = torch.get_rng_state()

try:
    _cfg_check = Config()
    _device_check = torch.device("cpu")

    _model_check = EigenPINN(
        _cfg_check
    ).to(
        _device_check
    )

    """
    Verify the S-S hard trial-function structure independently.
    
    Required:
    
        Phi = 0,
        Phi'' = 0
    
    at both endpoints, while Phi' must not be structurally constrained.
    """
    verify_ss_hard_constraint_structure(
        model=_model_check,
        cfg=_cfg_check,
        device=_device_check,
    )

    # Interior collocation points.
    _X_colloc_check = torch.linspace(
        _cfg_check.x_min,
        _cfg_check.x_max,
        18,
        dtype=torch.float64,
        device=_device_check,
    )[1:-1].reshape(-1, 1)

    # Lightweight Gauss-Legendre rule.
    _X_quad_check, _W_quad_check = gauss_legendre_tensors(
        n=16,
        device=_device_check,
        x_min=_cfg_check.x_min,
        x_max=_cfg_check.x_max,
    )

    # Real evaluation of the complete S-S physics-informed loss.
    _loss_check, _comp_check = compute_loss(
        model=_model_check,
        X_colloc_base=_X_colloc_check,
        X_quad=_X_quad_check,
        W_quad=_W_quad_check,
        cfg=_cfg_check,
        device=_device_check,
    )

    if not torch.isfinite(_loss_check).item():
        raise RuntimeError(
            "The S-S compute_loss test produced a non-finite total loss."
        )

    missing_components = (
        required_history_components
        - set(_comp_check.keys())
    )

    if missing_components:
        raise RuntimeError(
            "compute_loss output is missing required S-S components:\n"
            f"{sorted(missing_components)}"
        )

    # The exactly embedded S-S conditions should already be machine-zero.
    if float(
        _comp_check["hard_bc_max_abs"]
        .detach()
        .cpu()
    ) > 1.0e-9:
        raise RuntimeError(
            "S-S hard-boundary API check failed: "
            "Phi or Phi'' is not satisfied at machine precision."
        )

finally:
    torch.set_rng_state(
        _rng_state
    )


print(
    "compute_loss signature:",
    inspect.signature(compute_loss),
)

print(
    "S-S PINN loss API consistency check: OK"
)

2026-09-04 00:33:37,876 - INFO - S-S hard-constraint verification passed: Phi=Phi''=0 at both endpoints, while the trial structure does not impose Phi'=0.


compute_loss signature: (model: 'EigenPINN', X_colloc_base: 'torch.Tensor', X_quad: 'torch.Tensor', W_quad: 'torch.Tensor', cfg: 'Config', device: 'torch.device') -> 'Tuple[torch.Tensor, Dict[str, torch.Tensor]]'
S-S PINN loss API consistency check: OK


<div class="alert alert-danger" role="alert"> 
🔎 History/checkpoint utilities

In [ ]:
# History/checkpoint utilities

PHYSICS_HISTORY_KEYS = [
    # Total and PDE losses
    "total",
    "de",
    "de_raw",

    # Active S-S bending-moment boundary loss
    "bc",
    "bc_raw",

    # Exact differentiable normalization diagnostics
    "norm",
    "norm_raw",
    "norm_physical_raw",

    # Optional regularization and trainable eigenvalue
    "l2",
    "lambda",

    # r-weighted normalization diagnostics
    "mass_norm_integral",
    "raw_mass_norm_integral",

    # Governing-equation residual diagnostics
    "residual_rms",
    "operator_scale_rms",
    "relative_residual_rms",
    "max_abs_residual",

    # Left S-S endpoint diagnostics
    "phi_0",
    "dphi_0",
    "ddphi_0",
    "moment_core_0",
    "moment_0",

    # Right S-S endpoint diagnostics
    "phi_1",
    "dphi_1",
    "ddphi_1",
    "moment_core_1",
    "moment_1",

    # Aggregate S-S boundary diagnostics
    "moment_core_rms",
    "physical_moment_rms",
    "hard_bc_max_abs",

    # Additional normalized-mode diagnostics
    "phi_rms",
    "mass_operator_rms",
]


HISTORY_KEYS = [
    # Global hybrid-optimization bookkeeping
    "optimization_step",
    "optimizer_phase",
    "phase_step",

    # Physics / eigenpair diagnostics
    *PHYSICS_HISTORY_KEYS,

    # Optimizer diagnostics
    "lr_nn",
    "lr_lambda",
    "lr_lbfgs",
    "lbfgs_func_evals",
]


def make_history() -> Dict[str, list]:
    """
    Create an empty synchronized history dictionary for the complete
    Adam -> L-BFGS optimization trajectory.
    """
    return {
        key: []
        for key in HISTORY_KEYS
    }


def append_history(
    history: Dict[str, list],
    comp: Dict[str, torch.Tensor],
    optimization_step: int,
    optimizer_phase: str,
    phase_step: int,
    lr_nn: float = float("nan"),
    lr_lambda: float = float("nan"),
    lr_lbfgs: float = float("nan"),
    lbfgs_func_evals: int = 0,
) -> None:
    """
    Append one accepted optimization state to the synchronized history.

    Adam entries correspond to the explicitly evaluated training states used
    by the adopted Adam bookkeeping.

    L-BFGS entries are recorded only after an accepted optimizer block has
    completed; closure evaluations themselves are deliberately not appended
    because they are internal line-search / quasi-Newton trial states.

    Parameters
    ----------
    optimization_step
        Global hybrid-optimization coordinate. For L-BFGS this is

            adam_epochs + cumulative_LBFGS_iterations.

    optimizer_phase
        Either "Adam" or "L-BFGS".

    phase_step
        Epoch number within Adam or cumulative quasi-Newton iteration count
        within L-BFGS.

    lbfgs_func_evals
        Cumulative L-BFGS closure/function-evaluation count.
    """

    if optimizer_phase not in (
        "Adam",
        "L-BFGS",
    ):
        raise ValueError(
            "optimizer_phase must be either 'Adam' or 'L-BFGS'."
        )

    if optimization_step < 0:
        raise ValueError(
            "optimization_step must be nonnegative."
        )

    if phase_step < 0:
        raise ValueError(
            "phase_step must be nonnegative."
        )

    if lbfgs_func_evals < 0:
        raise ValueError(
            "lbfgs_func_evals must be nonnegative."
        )

    missing = (
        set(PHYSICS_HISTORY_KEYS)
        - set(comp.keys())
    )

    if missing:
        raise KeyError(
            "Cannot append history because the loss-component dictionary "
            "is missing required entries: "
            f"{sorted(missing)}"
        )

    # Hybrid optimization coordinates
    history[
        "optimization_step"
    ].append(
        int(
            optimization_step
        )
    )

    history[
        "optimizer_phase"
    ].append(
        str(
            optimizer_phase
        )
    )

    history[
        "phase_step"
    ].append(
        int(
            phase_step
        )
    )

    # Physics/eigenpair quantities
    for key in PHYSICS_HISTORY_KEYS:
        value = comp[
            key
        ]

        if not torch.is_tensor(
            value
        ):
            raise TypeError(
                f"History component '{key}' must be a torch.Tensor."
            )

        history[
            key
        ].append(
            float(
                value
                .detach()
                .cpu()
            )
        )

    # Optimizer quantities
    history[
        "lr_nn"
    ].append(
        float(
            lr_nn
        )
    )

    history[
        "lr_lambda"
    ].append(
        float(
            lr_lambda
        )
    )

    history[
        "lr_lbfgs"
    ].append(
        float(
            lr_lbfgs
        )
    )

    history[
        "lbfgs_func_evals"
    ].append(
        int(
            lbfgs_func_evals
        )
    )

    # Synchronization invariant
    lengths = {
        key: len(values)
        for key, values in history.items()
    }

    if len(
        set(
            lengths.values()
        )
    ) != 1:
        raise RuntimeError(
            "Training-history synchronization failure: "
            f"{lengths}"
        )


def cpu_state_dict(
    model: nn.Module,
) -> Dict[str, torch.Tensor]:
    """
    Return a completely detached CPU copy of the model state.

    This includes all trainable S-S quantities:
        - neural-network parameters,
        - slope_left,
        - slope_right,
        - lambda_raw.
    """
    return {
        key: (
            value
            .detach()
            .cpu()
            .clone()
        )
        for (
            key,
            value,
        ) in model.state_dict().items()
    }


def _to_cpu_recursive(
    obj,
):
    """
    Recursively copy tensors and nested optimizer/scheduler state objects
    to CPU for portable checkpoint storage.
    """
    if torch.is_tensor(
        obj
    ):
        return (
            obj
            .detach()
            .cpu()
            .clone()
        )

    if isinstance(
        obj,
        dict,
    ):
        return {
            key: _to_cpu_recursive(
                value
            )
            for (
                key,
                value,
            ) in obj.items()
        }

    if isinstance(
        obj,
        list,
    ):
        return [
            _to_cpu_recursive(
                value
            )
            for value in obj
        ]

    if isinstance(
        obj,
        tuple,
    ):
        return tuple(
            _to_cpu_recursive(
                value
            )
            for value in obj
        )

    return copy.deepcopy(
        obj
    )


def cpu_optimizer_state_dict(
    optimizer: torch.optim.Optimizer,
) -> dict:
    """
    Return a portable CPU copy of an optimizer state dictionary.
    """
    return _to_cpu_recursive(
        optimizer.state_dict()
    )


def save_history_csv(
    history: Dict[str, list],
    output_dir: Path,
) -> None:
    """
    Save the complete synchronized Adam -> L-BFGS history and each
    individual history component to CSV.
    """
    lengths = {
        key: len(values)
        for key, values in history.items()
    }

    if (
        lengths
        and len(
            set(
                lengths.values()
            )
        ) != 1
    ):
        raise RuntimeError(
            "Cannot save unsynchronized training history: "
            f"{lengths}"
        )

    history_frame = pd.DataFrame(
        history
    )

    history_frame.to_csv(
        output_dir
        / "training_history.csv",
        index=False,
    )

    for (
        key,
        values,
    ) in history.items():
        pd.DataFrame(
            {
                key: values
            }
        ).to_csv(
            output_dir
            / f"history_{key}.csv",
            index=False,
        )

<div class="alert alert-danger" role="alert"> 
🔎 Prediction and plotting

In [ ]:
# Prediction and plotting

PINN_COLOR = "purple"
REFERENCE_COLOR = "black"

# Publication-style colors for the physics-informed loss components
TOTAL_LOSS_COLOR = "#0B3C5D"
DE_LOSS_COLOR = "#C62828"
BC_LOSS_COLOR = "#2E7D32"

# Scalar convergence diagnostics
DIAGNOSTIC_COLOR = "black"

# Adam -> L-BFGS transition marker
PHASE_TRANSITION_COLOR = "0.45"


def predict(
    model: EigenPINN,
    X: torch.Tensor,
) -> np.ndarray:
    model.eval()

    with torch.no_grad():
        return (
            model(X)
            .detach()
            .cpu()
            .numpy()
            .reshape(-1)
        )


def maxabs_normalize_mode(
    y: np.ndarray,
) -> np.ndarray:
    """
    Normalize a mode shape by its maximum absolute amplitude.

    A deterministic sign convention is used only for visualization:
    the largest-magnitude point is made positive.
    """
    y = np.asarray(
        y,
        dtype=np.float64,
    ).reshape(-1)

    scale = float(
        np.max(
            np.abs(y)
        )
    )

    if scale < 1.0e-14:
        return y.copy()

    out = (
        y
        / scale
    )

    idx = int(
        np.argmax(
            np.abs(out)
        )
    )

    if out[idx] < 0.0:
        out = -out

    return out


def _align_reference_for_plot(
    phi_pinn: np.ndarray,
    phi_ref: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Max-absolute normalize PINN and reference modes and align their
    arbitrary global signs for visualization only.

    This operation does not enter training or any quantitative metric.
    """
    p = maxabs_normalize_mode(
        phi_pinn
    )

    q = maxabs_normalize_mode(
        phi_ref
    )

    if p.size != q.size:
        raise ValueError(
            "PINN and reference mode arrays must have identical sizes."
        )

    # Eigenfunctions are defined only up to an arbitrary global sign.
    if float(
        np.dot(
            p,
            q,
        )
    ) < 0.0:
        q = -q

    return p, q


def _optimizer_transition_step(
    history: Dict[str, list],
) -> Optional[float]:
    """
    Return the actual Adam-to-L-BFGS handoff step.

    The handoff occurs immediately after the final recorded Adam state and
    before the first accepted L-BFGS block-end state.
    """
    phases = history.get(
        "optimizer_phase",
        [],
    )

    steps = history.get(
        "optimization_step",
        [],
    )

    if len(phases) != len(steps):
        raise ValueError(
            "optimizer_phase and optimization_step history lengths "
            "are inconsistent."
        )

    adam_steps = [
        float(step)
        for (
            phase,
            step,
        ) in zip(
            phases,
            steps,
        )
        if str(
            phase
        ).strip().upper() == "ADAM"
    ]

    has_lbfgs = any(
        str(
            phase
        ).strip().upper() == "L-BFGS"
        for phase in phases
    )

    if (
        has_lbfgs
        and adam_steps
    ):
        return max(
            adam_steps
        )

    return None


def _add_optimizer_transition_marker(
    ax,
    transition_step: Optional[float],
) -> None:
    """
    Add the publication-style Adam -> L-BFGS transition marker.
    """
    if transition_step is None:
        return

    ax.axvline(
        transition_step,
        color=PHASE_TRANSITION_COLOR,
        linestyle=":",
        linewidth=1.25,
        alpha=0.90,
        zorder=0,
    )

    ax.text(
        transition_step,
        0.97,
        "Adam → L-BFGS",
        transform=ax.get_xaxis_transform(),
        fontsize=8.8,
        color=PHASE_TRANSITION_COLOR,
        rotation=90,
        verticalalignment="top",
        horizontalalignment="right",
    )


def plot_snapshot(
    model: EigenPINN,
    X_test: torch.Tensor,
    history: Dict[str, list],
    optimization_step: int,
    output_path: Path,
    reference_x: Optional[np.ndarray] = None,
    reference_phi: Optional[np.ndarray] = None,
    reference_lambda: Optional[float] = None,
) -> None:
    """
    Save one compact publication-oriented S-S hybrid-training snapshot.

    The four retained panels are:

        1. Eigenfunction Comparison
        2. Trainable Eigenvalue
        3. Physics-Informed Loss Components
        4. Relative PDE Residual RMS

    The horizontal training coordinate is the global optimization step,
    rather than an Adam-only epoch count.

    During L-BFGS, only accepted block-end states are shown. Internal
    strong-Wolfe closure evaluations are not part of the plotted history.

    For the present S-S branch:

        - Phi and Phi'' are hard-imposed at both endpoints.
        - The two bending-moment conditions remain active in BC Loss.

    If an independent BVP mode is supplied:

        PINN            : purple solid line
        Independent BVP : black dashed line

    BVP information is used only for visualization and does not enter
    optimization, loss evaluation, checkpoint selection, optimizer
    switching, or stopping.
    """
    if optimization_step < 1:
        raise ValueError(
            "optimization_step must be positive."
        )

    n_history = len(
        history[
            "total"
        ]
    )

    if n_history < 1:
        raise ValueError(
            "Training history is empty."
        )

    # Phase-aware history consistency
    required_plot_keys = (
        "optimization_step",
        "optimizer_phase",
        "total",
        "de",
        "bc",
        "lambda",
        "relative_residual_rms",
    )

    for key in required_plot_keys:
        if key not in history:
            raise KeyError(
                f"Required plotting history key is missing: {key}"
            )

        if len(
            history[
                key
            ]
        ) != n_history:
            raise ValueError(
                f"History length mismatch for key '{key}'."
            )

    steps = np.asarray(
        history[
            "optimization_step"
        ],
        dtype=np.float64,
    )

    if np.any(
        ~np.isfinite(
            steps
        )
    ):
        raise ValueError(
            "Non-finite optimization_step values detected."
        )

    if np.any(
        np.diff(
            steps
        ) <= 0.0
    ):
        raise ValueError(
            "optimization_step history must be strictly increasing."
        )

    transition_step = (
        _optimizer_transition_step(
            history
        )
    )

    # Current PINN eigenfunction
    x = (
        X_test
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )

    phi_pinn_raw = predict(
        model,
        X_test,
    )

    phi_pinn_plot = (
        maxabs_normalize_mode(
            phi_pinn_raw
        )
    )

    phi_ref_plot = None

    # Optional independent BVP reference
    if (
        reference_x is not None
        and reference_phi is not None
    ):
        reference_x = np.asarray(
            reference_x,
            dtype=np.float64,
        ).reshape(-1)

        reference_phi = np.asarray(
            reference_phi,
            dtype=np.float64,
        ).reshape(-1)

        if (
            reference_x.size != x.size
            or not np.allclose(
                reference_x,
                x,
                rtol=0.0,
                atol=1.0e-13,
            )
        ):
            raise ValueError(
                "Snapshot reference must be evaluated on the same "
                "X_test grid."
            )

        (
            phi_pinn_plot,
            phi_ref_plot,
        ) = _align_reference_for_plot(
            phi_pinn_raw,
            reference_phi,
        )

    # Figure
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(13.2, 9.0),
        constrained_layout=True,
    )

    fig.patch.set_facecolor(
        "white"
    )

    def format_axis(
        ax,
        grid: bool = True,
    ) -> None:
        ax.tick_params(
            axis="both",
            labelsize=10,
        )

        if grid:
            ax.grid(
                True,
                which="both",
                linestyle=":",
                linewidth=0.7,
                alpha=0.35,
            )

    # (a) Eigenfunction Comparison
    ax = axes[
        0,
        0,
    ]

    ax.plot(
        x,
        phi_pinn_plot,
        color=PINN_COLOR,
        linewidth=2.5,
        linestyle="-",
        label="PINN",
    )

    if phi_ref_plot is not None:
        ax.plot(
            x,
            phi_ref_plot,
            color=REFERENCE_COLOR,
            linewidth=2.1,
            linestyle="--",
            label="Independent BVP",
        )

    ax.set_title(
        "Eigenfunction Comparison",
        fontsize=12,
        fontweight="semibold",
    )

    ax.set_xlabel(
        r"$X=x/L$",
        fontsize=11,
    )

    ax.set_ylabel(
        r"$\frac{\Phi(X)}{\max\left|\Phi(X)\right|}$",
        fontsize=11,
        labelpad=10,
    )

    ax.set_xlim(
        float(
            x[0]
        ),
        float(
            x[-1]
        ),
    )

    ax.legend(
        loc="best",
        fontsize=9.5,
        frameon=True,
    )

    format_axis(
        ax
    )

    # (b) Trainable Eigenvalue
    ax = axes[
        0,
        1,
    ]

    lambda_vals = np.asarray(
        history[
            "lambda"
        ],
        dtype=np.float64,
    )

    ax.plot(
        steps,
        lambda_vals,
        color=DIAGNOSTIC_COLOR,
        linewidth=2.0,
    )

    _add_optimizer_transition_marker(
        ax,
        transition_step,
    )

    ax.set_title(
        "Trainable Eigenvalue",
        fontsize=12,
        fontweight="semibold",
    )

    ax.set_xlabel(
        "Optimization Step",
        fontsize=11,
    )

    ax.set_ylabel(
        r"$\lambda$",
        fontsize=11,
    )

    format_axis(
        ax
    )

    # (c) Physics-Informed Loss Components
    ax = axes[
        1,
        0,
    ]

    total_vals = np.asarray(
        history[
            "total"
        ],
        dtype=np.float64,
    )

    de_vals = np.asarray(
        history[
            "de"
        ],
        dtype=np.float64,
    )

    bc_vals = np.asarray(
        history[
            "bc"
        ],
        dtype=np.float64,
    )

    if (
        np.any(
            ~np.isfinite(
                total_vals
            )
        )
        or np.any(
            ~np.isfinite(
                de_vals
            )
        )
        or np.any(
            ~np.isfinite(
                bc_vals
            )
        )
    ):
        raise ValueError(
            "Non-finite loss-history values detected."
        )

    floor = 1.0e-300

    ax.plot(
        steps,
        np.maximum(
            total_vals,
            floor,
        ),
        color=TOTAL_LOSS_COLOR,
        linewidth=2.1,
        linestyle="-",
        label="Total Loss",
    )

    ax.plot(
        steps,
        np.maximum(
            de_vals,
            floor,
        ),
        color=DE_LOSS_COLOR,
        linewidth=1.9,
        linestyle="--",
        label="DE Loss",
    )

    # Unlike C-C, the S-S bending-moment BC contribution is active and
    # therefore plotted as a genuine loss component.
    ax.plot(
        steps,
        np.maximum(
            bc_vals,
            floor,
        ),
        color=BC_LOSS_COLOR,
        linewidth=1.9,
        linestyle="-.",
        label="BC Loss",
    )

    _add_optimizer_transition_marker(
        ax,
        transition_step,
    )

    ax.set_yscale(
        "log"
    )

    ax.set_title(
        "Physics-Informed Loss Components",
        fontsize=12,
        fontweight="semibold",
    )

    ax.set_xlabel(
        "Optimization Step",
        fontsize=11,
    )

    ax.set_ylabel(
        r"$\mathcal{L}$",
        fontsize=11,
    )

    ax.legend(
        loc="best",
        fontsize=9.5,
        frameon=True,
    )

    format_axis(
        ax
    )

    # (d) Relative PDE Residual RMS
    ax = axes[
        1,
        1,
    ]

    relative_residual = np.asarray(
        history[
            "relative_residual_rms"
        ],
        dtype=np.float64,
    )

    if np.any(
        ~np.isfinite(
            relative_residual
        )
    ):
        raise ValueError(
            "Non-finite relative PDE residual values detected."
        )

    ax.plot(
        steps,
        np.maximum(
            relative_residual,
            floor,
        ),
        color=DIAGNOSTIC_COLOR,
        linewidth=2.0,
    )

    _add_optimizer_transition_marker(
        ax,
        transition_step,
    )

    ax.set_yscale(
        "log"
    )

    ax.set_title(
        "Relative PDE Residual RMS",
        fontsize=12,
        fontweight="semibold",
    )

    ax.set_xlabel(
        "Optimization Step",
        fontsize=11,
    )

    ax.set_ylabel(
        r"$R_{\mathrm{rel,RMS}}$",
        fontsize=11,
    )

    format_axis(
        ax
    )

    fig.savefig(
        output_path,
        dpi=200,
        facecolor="white",
        transparent=False,
        bbox_inches="tight",
        pad_inches=0.16,
    )

    plt.close(
        fig
    )


def plot_final_mode_comparison(
    X: np.ndarray,
    phi_pinn: np.ndarray,
    lambda_pinn: float,
    optimization_step: int,
    output_path: Path,
    phi_ref: Optional[np.ndarray] = None,
    lambda_ref: Optional[float] = None,
) -> None:
    """
    Publication-style final S-S eigenfunction comparison.

    Purple solid line : PINN.
    Black dashed line : independent BVP.

    The global optimization-step number is intentionally not displayed
    in the final publication-oriented figure.
    """
    if optimization_step < 1:
        raise ValueError(
            "optimization_step must be positive."
        )

    X = np.asarray(
        X,
        dtype=np.float64,
    ).reshape(-1)

    phi_pinn = np.asarray(
        phi_pinn,
        dtype=np.float64,
    ).reshape(-1)

    if X.size != phi_pinn.size:
        raise ValueError(
            "X and phi_pinn must have identical sizes."
        )

    phi_pinn_plot = (
        maxabs_normalize_mode(
            phi_pinn
        )
    )

    phi_ref_plot = None

    if phi_ref is not None:
        phi_ref = np.asarray(
            phi_ref,
            dtype=np.float64,
        ).reshape(-1)

        if phi_ref.size != X.size:
            raise ValueError(
                "phi_ref must be evaluated on the same grid as phi_pinn."
            )

        (
            phi_pinn_plot,
            phi_ref_plot,
        ) = _align_reference_for_plot(
            phi_pinn,
            phi_ref,
        )

    fig, ax = plt.subplots(
        figsize=(9.2, 6.2),
        constrained_layout=True,
    )

    fig.patch.set_facecolor(
        "white"
    )

    ax.plot(
        X,
        phi_pinn_plot,
        color=PINN_COLOR,
        linewidth=2.8,
        linestyle="-",
        label="PINN",
    )

    if phi_ref_plot is not None:
        ax.plot(
            X,
            phi_ref_plot,
            color=REFERENCE_COLOR,
            linewidth=2.3,
            linestyle="--",
            label="Independent BVP",
        )

    ax.set_title(
        "Eigenfunction Comparison",
        fontsize=14,
        fontweight="semibold",
        pad=12,
    )

    ax.set_xlabel(
        r"$X=x/L$",
        fontsize=12,
        labelpad=7,
    )

    ax.set_ylabel(
        r"$\frac{\Phi(X)}{\max\left|\Phi(X)\right|}$",
        fontsize=12,
        labelpad=10,
    )

    ax.set_xlim(
        float(
            X[0]
        ),
        float(
            X[-1]
        ),
    )

    ax.tick_params(
        axis="both",
        labelsize=11,
    )

    ax.grid(
        True,
        linestyle=":",
        linewidth=0.75,
        alpha=0.35,
    )

    ax.legend(
        loc="best",
        fontsize=11,
        frameon=True,
    )

    # Display only the PINN eigenvalue; the independent BVP eigenvalue
    # remains a validation quantity and is not annotated in this figure.
    ax.text(
        0.03,
        0.96,
        rf"$\lambda_{{\rm PINN}} = {lambda_pinn:.6f}$",
        transform=ax.transAxes,
        fontsize=10.5,
        verticalalignment="top",
        horizontalalignment="left",
        bbox={
            "boxstyle": "round,pad=0.35",
            "facecolor": "white",
            "edgecolor": "0.70",
            "alpha": 0.92,
        },
    )

    fig.savefig(
        output_path,
        dpi=220,
        facecolor="white",
        transparent=False,
        bbox_inches="tight",
        pad_inches=0.18,
    )

    plt.close(
        fig
    )


def save_coefficient_profiles(
    cfg: Config,
    output_dir: Path,
) -> None:
    X = np.linspace(
        0.0,
        1.0,
        501,
    )

    sec = section_resultants_numpy(
        X,
        cfg,
    )

    df = pd.DataFrame(
        {
            "X": X,
            **sec,
        }
    )

    df[
        "zE_over_h"
    ] = (
        df[
            "zE"
        ]
        / cfg.h
    )

    df[
        "zrho_over_h"
    ] = (
        df[
            "zrho"
        ]
        / cfg.h
    )

    df.to_csv(
        output_dir
        / "coefficient_profiles.csv",
        index=False,
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(11.5, 4.8),
        constrained_layout=True,
    )

    fig.patch.set_facecolor(
        "white"
    )

    # Dimensionless coefficient profiles
    ax1 = axes[
        0
    ]

    ax1.plot(
        X,
        sec[
            "d"
        ],
        linewidth=2.0,
        label=r"$d(X)=D^\star/D_0$",
    )

    ax1.plot(
        X,
        sec[
            "r"
        ],
        linewidth=2.0,
        label=r"$r(X)=I_0^\rho/m_0$",
    )

    ax1.set_xlabel(
        r"$X=x/L$",
        fontsize=11,
    )

    ax1.set_title(
        "Dimensionless Coefficient Profiles",
        fontsize=12,
        fontweight="semibold",
    )

    ax1.grid(
        True,
        linestyle=":",
        linewidth=0.7,
        alpha=0.35,
    )

    ax1.legend(
        fontsize=10,
        frameon=True,
    )

    # Elastic neutral axis / mass centroid
    ax2 = axes[
        1
    ]

    ax2.plot(
        X,
        sec[
            "zE"
        ]
        / cfg.h,
        linewidth=2.0,
        label="Elastic neutral axis",
    )

    ax2.plot(
        X,
        sec[
            "zrho"
        ]
        / cfg.h,
        linewidth=2.0,
        label="Mass centroid",
    )

    ax2.set_xlabel(
        r"$X=x/L$",
        fontsize=11,
    )

    ax2.set_ylabel(
        r"$z/h$",
        fontsize=11,
    )

    ax2.set_title(
        "Elastic Neutral Axis and Mass Centroid",
        fontsize=12,
        fontweight="semibold",
    )

    ax2.grid(
        True,
        linestyle=":",
        linewidth=0.7,
        alpha=0.35,
    )

    ax2.legend(
        fontsize=10,
        frameon=True,
    )

    fig.savefig(
        output_dir
        / "coefficient_profiles.png",
        dpi=180,
        facecolor="white",
        transparent=False,
        bbox_inches="tight",
        pad_inches=0.16,
    )

    plt.close(
        fig
    )

<div class="alert alert-danger" role="alert"> 
🔎 Physical frequency conversion

In [ ]:
# Physical frequency conversion

def lambda_to_omega(lambda_value: float, cfg: Config) -> float:
    return math.sqrt(lambda_value * cfg.D0 / (cfg.m0 * cfg.L**4))


def lambda_to_frequency_hz(lambda_value: float, cfg: Config) -> float:
    return lambda_to_omega(lambda_value, cfg) / (2.0 * math.pi)

<div class="alert alert-danger" role="alert"> 
🔎 Independent post-training BVP reference solver

In [ ]:
# Independent S-S BVP reference solver and eigenmode-comparison metrics

def _falling_power_derivative(
    X: np.ndarray,
    exponent: float,
    order: int,
) -> np.ndarray:
    """
    Compute

        d^order / dX^order (X^exponent).

    Integer powers are treated explicitly so that derivatives whose order
    exceeds the polynomial degree return exact zeros instead of expressions
    such as

        0 * X^negative_power.
    """
    X = np.asarray(
        X,
        dtype=np.float64,
    )

    if order < 0:
        raise ValueError(
            "Derivative order must be nonnegative."
        )

    if order == 0:
        return np.power(
            X,
            exponent,
        )

    is_integer = (
        abs(
            exponent
            - round(exponent)
        )
        < 1.0e-12
    )

    n_int = (
        int(
            round(exponent)
        )
        if is_integer
        else None
    )

    if (
        is_integer
        and order > n_int
    ):
        return np.zeros_like(
            X
        )

    coeff = 1.0

    for j in range(order):
        coeff *= (
            exponent
            - j
        )

    if abs(coeff) < 1.0e-15:
        return np.zeros_like(
            X
        )

    with np.errstate(
        divide="ignore",
        invalid="ignore",
        over="ignore",
    ):
        result = (
            coeff
            * np.power(
                X,
                exponent - order,
            )
        )

    return result


def coefficient_derivatives_numpy(
    X: np.ndarray,
    cfg: Config,
) -> Tuple[np.ndarray, ...]:
    """
    Return

        d, d', d'', d''', r, r', r''

    analytically for the independent expanded-form BVP solver.

    The formulas are mathematically equivalent to the coefficient functions
    used by the PINN, but are evaluated independently with NumPy rather than
    PyTorch automatic differentiation.
    """
    X = np.asarray(
        X,
        dtype=np.float64,
    )

    q = _falling_power_derivative(
        X,
        cfg.k,
        0,
    )

    q1 = _falling_power_derivative(
        X,
        cfg.k,
        1,
    )

    q2 = _falling_power_derivative(
        X,
        cfg.k,
        2,
    )

    q3 = _falling_power_derivative(
        X,
        cfg.k,
        3,
    )

    cA, cB, cD = _section_constants(
        cfg
    )

    eta_E = (
        (cfg.E_m - cfg.E_c)
        / cfg.E_c
    )

    eta_rho = (
        (cfg.rho_m - cfg.rho_c)
        / cfg.rho_c
    )

    """
    Dimensionless condensed flexural stiffness
    
        d(q)
          = 1
            + A q
            - B q^2 / (1 + C q).
    """
    A = (
        12.0
        * eta_E
        * cD
    )

    B = (
        12.0
        * eta_E**2
        * cB**2
    )

    C = (
        eta_E
        * cA
    )

    den = (
        1.0
        + C * q
    )

    if (
        np.any(
            ~np.isfinite(
                den
            )
        )
        or np.any(
            np.abs(
                den
            )
            < 1.0e-14
        )
    ):
        raise ValueError(
            "Invalid denominator encountered in dimensionless stiffness."
        )

    g = (
        q**2
        / den
    )

    # Derivatives of
    # g(q) = q^2 / (1 + C q)
    gq = (
        q
        * (
            C * q
            + 2.0
        )
        / den**2
    )

    gqq = (
        2.0
        / den**3
    )

    gqqq = (
        -6.0
        * C
        / den**4
    )

    # Derivatives of d with respect to q.
    fq = (
        A
        - B * gq
    )

    fqq = (
        -B
        * gqq
    )

    fqqq = (
        -B
        * gqqq
    )

    # Chain-rule derivatives with respect to X.
    d = (
        1.0
        + A * q
        - B * g
    )

    d1 = (
        fq
        * q1
    )

    d2 = (
        fqq * q1**2
        + fq * q2
    )

    d3 = (
        fqqq * q1**3
        + 3.0
        * fqq
        * q1
        * q2
        + fq
        * q3
    )

    # Dimensionless translational mass coefficient
    # r(X) = 1 + R X^k.
    R = (
        eta_rho
        * cA
    )

    r = (
        1.0
        + R * q
    )

    r1 = (
        R
        * q1
    )

    r2 = (
        R
        * q2
    )

    return (
        d,
        d1,
        d2,
        d3,
        r,
        r1,
        r2,
    )


def _base_ss_sine_derivative(
    X: np.ndarray,
    order: int,
) -> np.ndarray:
    """
    Derivatives of the simply-supported first-mode-like initialization

        Phi_0(X) = sin(pi X).

    Analytically,

        Phi_0^(n)(X)
            =
        pi^n sin(pi X + n*pi/2).

    This initialization satisfies

        Phi_0(0)   = Phi_0''(0) = 0,
        Phi_0(1)   = Phi_0''(1) = 0,

    while keeping the endpoint rotations nonzero.

    It is used only as an initial guess for the independent SciPy BVP
    solver. It is not required to satisfy the complete NSGT bending-moment
    conditions before solve_bvp starts.
    """
    X = np.asarray(
        X,
        dtype=np.float64,
    )

    if order < 0:
        raise ValueError(
            "Derivative order must be nonnegative."
        )

    return (
        math.pi**order
        * np.sin(
            math.pi * X
            + order
            * math.pi
            / 2.0
        )
    )


def reference_bvp_is_regular(
    cfg: Config,
) -> bool:
    """
    Check endpoint regularity of the expanded independent BVP coefficients.

    The expanded sixth-order equation requires d''' and r''.

    For noninteger k < 3, these coefficient derivatives can be singular
    at X=0. Such cases are therefore excluded from this particular
    expanded-form reference solver.
    """
    is_integer_k = (
        abs(
            cfg.k
            - round(cfg.k)
        )
        < 1.0e-12
    )

    return (
        is_integer_k
        or cfg.k >= 3.0
    )


def count_interior_nodes(
    X: np.ndarray,
    phi: np.ndarray,
    relative_tol: float = 1.0e-7,
) -> int:
    """
    Count sign-changing interior nodes of an eigenfunction.

    Boundary zeros are excluded. Values satisfying

        |Phi| <= relative_tol * max|Phi|

    are ignored so that numerical roundoff near a zero is not interpreted
    as an additional physical node.
    """
    X = np.asarray(
        X,
        dtype=np.float64,
    ).reshape(-1)

    phi = np.asarray(
        phi,
        dtype=np.float64,
    ).reshape(-1)

    if X.size != phi.size:
        raise ValueError(
            "X and phi must have identical sizes."
        )

    if X.size < 3:
        return 0

    if not np.all(
        np.isfinite(
            phi
        )
    ):
        raise ValueError(
            "Non-finite mode-shape values encountered."
        )

    amplitude = float(
        np.max(
            np.abs(
                phi
            )
        )
    )

    if amplitude < 1.0e-14:
        return 0

    tol = (
        relative_tol
        * amplitude
    )

    # Exclude the two physical boundaries.
    phi_int = (
        phi[
            1:-1
        ]
    )

    # Ignore values numerically indistinguishable from zero.
    phi_sig = (
        phi_int[
            np.abs(
                phi_int
            )
            > tol
        ]
    )

    if phi_sig.size < 2:
        return 0

    signs = np.sign(
        phi_sig
    )

    return int(
        np.sum(
            signs[:-1]
            * signs[1:]
            < 0.0
        )
    )


def solve_reference_bvp_ss(
    cfg: Config,
) -> Optional[dict]:
    """
    Independently solve the same reduced-flexural sixth-order S-S
    Euler-Bernoulli-NSGT eigenproblem using SciPy solve_bvp.

    This BVP solution is a numerical reference only.

    It must not enter

        - the PINN objective,
        - Adam optimization,
        - L-BFGS optimization,
        - learning-rate scheduling,
        - Adam -> L-BFGS switching,
        - checkpoint selection,
        - optimizer stopping criteria.

    It may be evaluated independently for validation and for optional
    visualization of the current PINN mode.

    -------------------------------------------------------------------------
    BVP state
    -------------------------------------------------------------------------

        Y = [
            Phi,
            Phi',
            Phi'',
            Phi''',
            Phi'''',
            Phi''''',
            I
        ]^T,

    where

        I'(X) = r(X) Phi(X)^2.

    Lambda is one unknown BVP parameter.

    -------------------------------------------------------------------------
    S-S boundary conditions
    -------------------------------------------------------------------------

    At each endpoint:

        Phi = 0,
        Phi'' = 0,
        M_bar = 0,

    with

        M_bar
            =
        -d Phi''
        + zeta^2 [d' Phi''' + d Phi'''']
        - tau^2 lambda r Phi.

    The independent r-weighted normalization is

        I(0) = 0,
        I(1) = 1.

    Therefore seven first-order state variables plus one unknown eigenvalue
    parameter are supplied with eight independent boundary conditions.
    """
    if not cfg.run_reference_bvp:
        return None

    if not SCIPY_AVAILABLE:
        LOGGER.warning(
            "SciPy unavailable: skipping independent S-S BVP validation."
        )
        return None

    if not reference_bvp_is_regular(
        cfg
    ):
        LOGGER.warning(
            "Skipping S-S BVP validation because noninteger k < 3 "
            "produces endpoint-singular coefficient derivatives in the "
            "expanded strong form."
        )
        return None

    if cfg.zeta <= 0.0:
        LOGGER.warning(
            "Skipping sixth-order S-S BVP validation because zeta "
            "must be strictly positive."
        )
        return None

    # The present nondimensional model is defined over X in [0,1].
    if (
        not math.isclose(
            cfg.x_min,
            0.0,
            rel_tol=0.0,
            abs_tol=1.0e-14,
        )
        or not math.isclose(
            cfg.x_max,
            1.0,
            rel_tol=0.0,
            abs_tol=1.0e-14,
        )
    ):
        LOGGER.warning(
            "Independent S-S BVP validation currently requires X in [0,1]; "
            "reference solution skipped."
        )
        return None

    x = np.linspace(
        cfg.x_min,
        cfg.x_max,
        201,
        dtype=np.float64,
    )

    (
        d0,
        _,
        _,
        _,
        r0,
        _,
        _,
    ) = coefficient_derivatives_numpy(
        x,
        cfg,
    )

    if (
        np.any(
            ~np.isfinite(
                d0
            )
        )
        or np.any(
            ~np.isfinite(
                r0
            )
        )
        or np.min(
            d0
        ) <= 0.0
        or np.min(
            r0
        ) <= 0.0
    ):
        LOGGER.warning(
            "Invalid stiffness or mass coefficients detected; "
            "S-S BVP validation skipped."
        )
        return None

    # First-mode-like initialization
    # Phi_0(X) = sin(pi X).
    base = _base_ss_sine_derivative(
        x,
        order=0,
    )

    norm0 = np.trapezoid(
        r0
        * base**2,
        x,
    )

    if (
        not np.isfinite(
            norm0
        )
        or norm0 <= 1.0e-300
    ):
        LOGGER.warning(
            "Invalid initial S-S BVP normalization integral."
        )
        return None

    scale = (
        1.0
        / math.sqrt(
            norm0
        )
    )

    """
    Seven first-order states:
    
        y0 = Phi
        y1 = Phi'
        y2 = Phi''
        y3 = Phi'''
        y4 = Phi''''
        y5 = Phi'''''
        y6 = integral r Phi^2 dX
    """
    y0 = np.zeros(
        (
            7,
            x.size,
        ),
        dtype=np.float64,
    )

    for j in range(6):
        y0[
            j,
            :,
        ] = (
            scale
            * _base_ss_sine_derivative(
                x,
                order=j,
            )
        )

    y0[
        6,
        :,
    ] = cumulative_trapezoid(
        r0
        * y0[
            0
        ]**2,
        x,
        initial=0.0,
    )

    # Expanded first-order ODE system
    def ode(
        X: np.ndarray,
        Y: np.ndarray,
        p: np.ndarray,
    ) -> np.ndarray:
        lam = float(
            p[
                0
            ]
        )

        (
            d,
            d1,
            d2,
            d3,
            r,
            r1,
            r2,
        ) = coefficient_derivatives_numpy(
            X,
            cfg,
        )

        (
            phi,
            phi1,
            phi2,
            phi3,
            phi4,
            phi5,
            _,
        ) = Y

        # Generalized nonlocal mass operator
        mass_operator = (
            r * phi
            - cfg.tau**2
            * (
                r * phi2
                + 2.0
                * r1
                * phi1
                + r2
                * phi
            )
        )

        # Expanded governing equation
        numerator_phi6 = (
            d
            * phi4

            + 2.0
            * d1
            * phi3

            + d2
            * phi2

            - cfg.zeta**2
            * (
                3.0
                * d1
                * phi5

                + 3.0
                * d2
                * phi4

                + d3
                * phi3
            )

            - lam
            * mass_operator
        )

        phi6 = (
            numerator_phi6
            / (
                cfg.zeta**2
                * d
            )
        )

        return np.vstack(
            (
                phi1,
                phi2,
                phi3,
                phi4,
                phi5,
                phi6,
                r * phi**2,
            )
        )

    # Endpoint coefficients for the physical S-S moment conditions
    X_end = np.array(
        [
            cfg.x_min,
            cfg.x_max,
        ],
        dtype=np.float64,
    )

    (
        d_end,
        d1_end,
        _,
        _,
        r_end,
        _,
        _,
    ) = coefficient_derivatives_numpy(
        X_end,
        cfg,
    )

    if (
        not np.all(
            np.isfinite(
                d_end
            )
        )
        or not np.all(
            np.isfinite(
                d1_end
            )
        )
        or not np.all(
            np.isfinite(
                r_end
            )
        )
    ):
        LOGGER.warning(
            "Non-finite endpoint coefficients detected; "
            "S-S BVP validation skipped."
        )
        return None

    # Li-type S-S BCs + independent integral normalization
    def bc(
        ya: np.ndarray,
        yb: np.ndarray,
        p: np.ndarray,
    ) -> np.ndarray:
        lam = float(
            p[
                0
            ]
        )

        # Complete dimensionless bending moment:
        moment_a = (
            -d_end[
                0
            ]
            * ya[
                2
            ]

            + cfg.zeta**2
            * (
                d1_end[
                    0
                ]
                * ya[
                    3
                ]

                + d_end[
                    0
                ]
                * ya[
                    4
                ]
            )

            - cfg.tau**2
            * lam
            * r_end[
                0
            ]
            * ya[
                0
            ]
        )

        moment_b = (
            -d_end[
                1
            ]
            * yb[
                2
            ]

            + cfg.zeta**2
            * (
                d1_end[
                    1
                ]
                * yb[
                    3
                ]

                + d_end[
                    1
                ]
                * yb[
                    4
                ]
            )

            - cfg.tau**2
            * lam
            * r_end[
                1
            ]
            * yb[
                0
            ]
        )

        # Eight conditions:
        return np.array(
            [
                ya[
                    0
                ],
                ya[
                    2
                ],
                moment_a,

                yb[
                    0
                ],
                yb[
                    2
                ],
                moment_b,

                ya[
                    6
                ],
                yb[
                    6
                ]
                - 1.0,
            ],
            dtype=np.float64,
        )

    # Independent eigenvalue guesses
    guess_factors = (
        0.5,
        1.0,
        2.0,
    )

    guesses = [
        factor
        * cfg.reference_lambda_guess
        for factor in guess_factors
    ]

    candidates = []

    # Dense grid used only to characterize independently converged modes.
    x_check = np.linspace(
        cfg.x_min,
        cfg.x_max,
        2001,
        dtype=np.float64,
    )

    for guess in guesses:
        try:
            sol = solve_bvp(
                ode,
                bc,
                x,
                y0.copy(),
                p=np.array(
                    [
                        guess
                    ],
                    dtype=np.float64,
                ),
                tol=cfg.reference_tol,
                max_nodes=cfg.reference_max_nodes,
                verbose=0,
            )

        except Exception as exc:
            LOGGER.warning(
                "S-S BVP attempt with lambda guess %.6g failed: %s",
                guess,
                exc,
            )
            continue

        if (
            sol.status != 0
            or sol.p is None
            or not np.isfinite(
                sol.p[
                    0
                ]
            )
            or sol.p[
                0
            ] <= 0.0
            or not np.all(
                np.isfinite(
                    sol.y
                )
            )
        ):
            continue

        phi_candidate = (
            sol.sol(
                x_check
            )[
                0
            ]
        )

        if not np.all(
            np.isfinite(
                phi_candidate
            )
        ):
            continue

        n_nodes = count_interior_nodes(
            x_check,
            phi_candidate,
        )

        candidates.append(
            {
                "solution": sol,
                "lambda": float(
                    sol.p[
                        0
                    ]
                ),
                "interior_nodes": int(
                    n_nodes
                ),
                "initial_guess": float(
                    guess
                ),
            }
        )

        LOGGER.info(
            "S-S BVP candidate: guess=%.6g | lambda=%.12f | "
            "interior_nodes=%d | mesh_nodes=%d | max_rms=%.3e",
            guess,
            float(
                sol.p[
                    0
                ]
            ),
            n_nodes,
            sol.x.size,
            float(
                np.max(
                    sol.rms_residuals
                )
            ),
        )

    if not candidates:
        LOGGER.warning(
            "Independent S-S BVP solver did not converge "
            "to a positive eigenpair."
        )
        return None

    # Fundamental-mode identification
    first_mode_candidates = [
        candidate
        for candidate in candidates
        if candidate[
            "interior_nodes"
        ] == 0
    ]

    if not first_mode_candidates:
        LOGGER.warning(
            "Positive S-S BVP eigenpairs were obtained, but none could be "
            "identified as a zero-interior-node fundamental-mode candidate."
        )
        return None

    selected = min(
        first_mode_candidates,
        key=lambda item: item[
            "lambda"
        ],
    )

    sol = selected[
        "solution"
    ]

    lam = float(
        selected[
            "lambda"
        ]
    )

    # Independent dense-grid r-weighted normalization verification
    phi_check = (
        sol.sol(
            x_check
        )[
            0
        ]
    )

    r_check = (
        section_resultants_numpy(
            x_check,
            cfg,
        )[
            "r"
        ]
    )

    r_weighted_norm_check = np.trapezoid(
        r_check
        * phi_check**2,
        x_check,
    )

    # Independent verification of physical endpoint moments
    ya_check = (
        sol.sol(
            np.array(
                [
                    cfg.x_min
                ],
                dtype=np.float64,
            )
        )[
            :,
            0,
        ]
    )

    yb_check = (
        sol.sol(
            np.array(
                [
                    cfg.x_max
                ],
                dtype=np.float64,
            )
        )[
            :,
            0,
        ]
    )

    moment_0_check = (
        -d_end[
            0
        ]
        * ya_check[
            2
        ]

        + cfg.zeta**2
        * (
            d1_end[
                0
            ]
            * ya_check[
                3
            ]

            + d_end[
                0
            ]
            * ya_check[
                4
            ]
        )

        - cfg.tau**2
        * lam
        * r_end[
            0
        ]
        * ya_check[
            0
        ]
    )

    moment_1_check = (
        -d_end[
            1
        ]
        * yb_check[
            2
        ]

        + cfg.zeta**2
        * (
            d1_end[
                1
            ]
            * yb_check[
                3
            ]

            + d_end[
                1
            ]
            * yb_check[
                4
            ]
        )

        - cfg.tau**2
        * lam
        * r_end[
            1
        ]
        * yb_check[
            0
        ]
    )

    max_rms_residual = float(
        np.max(
            sol.rms_residuals
        )
    )

    LOGGER.info(
        "Independent first-mode S-S BVP validation converged: "
        "lambda_ref=%.12f | interior_nodes=%d | mesh_nodes=%d | "
        "max_rms=%.3e | r_weighted_norm=%.12f | "
        "M0=%.3e | M1=%.3e",
        lam,
        selected[
            "interior_nodes"
        ],
        sol.x.size,
        max_rms_residual,
        float(
            r_weighted_norm_check
        ),
        float(
            moment_0_check
        ),
        float(
            moment_1_check
        ),
    )

    return {
        "solution": sol,

        "lambda": lam,

        "mode_number": 1,

        "interior_nodes": int(
            selected[
                "interior_nodes"
            ]
        ),

        "initial_guess": float(
            selected[
                "initial_guess"
            ]
        ),

        "r_weighted_norm_integral": float(
            r_weighted_norm_check
        ),

        "moment_0": float(
            moment_0_check
        ),

        "moment_1": float(
            moment_1_check
        ),

        "mesh_nodes": int(
            sol.x.size
        ),

        "max_rms_residual": max_rms_residual,
    }


def weighted_mode_metrics(
    X: np.ndarray,
    phi_pinn: np.ndarray,
    phi_ref: np.ndarray,
    cfg: Config,
) -> Dict[str, float]:
    """
    Compare PINN and independent-reference eigenfunctions using the
    r(X)-weighted inner product

        <u,v>_r
            =
        integral r(X) u(X) v(X) dX.

    Both modes are independently normalized using

        integral r(X) Phi(X)^2 dX = 1

    before sign alignment.

    The reported MAC is therefore specifically an r-weighted MAC, not a
    full generalized mass-matrix MAC.
    """
    X = np.asarray(
        X,
        dtype=np.float64,
    ).reshape(-1)

    p = np.asarray(
        phi_pinn,
        dtype=np.float64,
    ).reshape(-1)

    q = np.asarray(
        phi_ref,
        dtype=np.float64,
    ).reshape(-1)

    if not (
        X.size
        == p.size
        == q.size
    ):
        raise ValueError(
            "X, phi_pinn, and phi_ref must have identical sizes."
        )

    if X.size < 2:
        raise ValueError(
            "At least two spatial points are required."
        )

    if np.any(
        np.diff(
            X
        )
        <= 0.0
    ):
        raise ValueError(
            "X must be strictly increasing."
        )

    if (
        not np.all(
            np.isfinite(
                X
            )
        )
        or not np.all(
            np.isfinite(
                p
            )
        )
        or not np.all(
            np.isfinite(
                q
            )
        )
    ):
        raise ValueError(
            "Non-finite values encountered in mode-shape comparison."
        )

    r = (
        section_resultants_numpy(
            X,
            cfg,
        )[
            "r"
        ]
    )

    if (
        not np.all(
            np.isfinite(
                r
            )
        )
        or np.min(
            r
        ) <= 0.0
    ):
        raise ValueError(
            "Invalid r-weighting coefficient r(X)."
        )

    # Independent r-weighted normalization
    norm_p_sq = np.trapezoid(
        r
        * p**2,
        X,
    )

    norm_q_sq = np.trapezoid(
        r
        * q**2,
        X,
    )

    if (
        not np.isfinite(
            norm_p_sq
        )
        or norm_p_sq <= 1.0e-300
    ):
        raise ValueError(
            "PINN eigenfunction has an invalid r-weighted norm."
        )

    if (
        not np.isfinite(
            norm_q_sq
        )
        or norm_q_sq <= 1.0e-300
    ):
        raise ValueError(
            "Reference eigenfunction has an invalid r-weighted norm."
        )

    p = (
        p
        / math.sqrt(
            norm_p_sq
        )
    )

    q = (
        q
        / math.sqrt(
            norm_q_sq
        )
    )

    # Global sign alignment
    inner = np.trapezoid(
        r
        * p
        * q,
        X,
    )

    if inner < 0.0:
        q = -q
        inner = -inner

    # r-weighted Modal Assurance Criterion
    mac = float(
        inner**2
    )

    # Protect against tiny quadrature roundoff outside the exact [0,1] range.
    mac = min(
        max(
            mac,
            0.0,
        ),
        1.0,
    )

    # r-weighted relative L2 mode-shape error
    error_sq = np.trapezoid(
        r
        * (
            p
            - q
        )**2,
        X,
    )

    error_sq = max(
        float(
            error_sq
        ),
        0.0,
    )

    rel_l2 = math.sqrt(
        error_sq
    )

    return {
        "r_weighted_MAC": float(
            mac
        ),

        "relative_L2_mode_error": float(
            rel_l2
        ),
    }

<div class="alert alert-danger" role="alert"> 
🔎 Training

In [ ]:
# Training

def train(cfg: Config) -> Dict[str, object]:
    """
    Train the unsupervised eigenvalue PINN for the Reduced Flexural
    2D-FG Euler-Bernoulli-NSGT nanobeam with the adopted Li-type S-S
    fixed-curvature boundary conditions.

    S-S boundary conditions
    -----------------------

    At both endpoints,

        Phi = 0,
        Phi'' = 0,
        M_bar = 0.

    The hard-constrained trial architecture imposes

        Phi = 0,
        Phi'' = 0

    exactly, while the two bending-moment conditions remain active
    physics-informed boundary constraints.

    Optimization protocol
    ---------------------

    Stage I
        Adam for cfg.adam_epochs epochs.

    Stage II
        Deterministic full-batch L-BFGS refinement for up to
        cfg.lbfgs_max_iter quasi-Newton iterations.

    The Adam -> L-BFGS handoff is prescribed a priori and is not triggered
    by any BVP/reference information.

    The same fixed collocation points, quadrature points, governing
    residual, S-S boundary residuals, and differentiable r-weighted
    normalization are used in both optimizer phases.

    The independent SciPy BVP solution is a numerical validation/reference
    calculation only. It never enters

        - the PINN objective,
        - Adam updates,
        - L-BFGS updates,
        - learning-rate scheduling,
        - optimizer switching,
        - checkpoint selection,
        - optimizer stopping criteria.
    """

    # Configuration, reproducibility, and device
    validate_config(
        cfg
    )

    set_seed(
        cfg.seed,
        cfg.deterministic_torch,
    )

    device = resolve_device(
        cfg.device
    )

    # The present nondimensional formulation is defined on X=x/L in [0,1].
    if not (
        math.isclose(
            cfg.x_min,
            0.0,
            rel_tol=0.0,
            abs_tol=1.0e-14,
        )
        and math.isclose(
            cfg.x_max,
            1.0,
            rel_tol=0.0,
            abs_tol=1.0e-14,
        )
    ):
        raise ValueError(
            "The present 2D-FG EB-NSGT formulation is defined on "
            "the nondimensional domain X=x/L in [0,1]. "
            "Use x_min=0 and x_max=1."
        )

    LOGGER.info(
        "Using device: %s",
        device,
    )

    LOGGER.info(
        "Default dtype: %s",
        torch.get_default_dtype(),
    )

    LOGGER.info(
        "Model: Reduced Flexural 2D-FG Euler-Bernoulli-NSGT, "
        "Li-type S-S fixed-curvature BC"
    )

    LOGGER.info(
        "k=%.6g, beta=%.6g, tau=%.6g, zeta=%.6g, L/h=%.6g",
        cfg.k,
        cfg.beta,
        cfg.tau,
        cfg.zeta,
        cfg.L_over_h,
    )

    LOGGER.info(
        "Optimization protocol: Adam=%d epochs -> "
        "L-BFGS<=%d quasi-Newton iterations",
        cfg.adam_epochs,
        cfg.lbfgs_max_iter,
    )

    LOGGER.info(
        "S-S objective: hard Phi=Phi''=0 at both endpoints; "
        "active endpoint bending-moment BC loss."
    )

    # Output directories
    output_dir = Path(
        cfg.output_dir
    )

    if output_dir.exists():
        if any(
            output_dir.iterdir()
        ):
            raise FileExistsError(
                f"Output directory already contains files: {output_dir}\n"
                "Use a new output_dir for a clean independent run."
            )
    else:
        output_dir.mkdir(
            parents=True,
            exist_ok=False,
        )

    plot_dir = (
        output_dir
        / "optimization_plots"
    )

    if (
        cfg.save_epoch_plots
        and cfg.plot_every > 0
    ):
        plot_dir.mkdir(
            parents=True,
            exist_ok=False,
        )

    # Save configuration and derived dimensional quantities
    config_dict = asdict(
        cfg
    )

    config_dict[
        "derived"
    ] = {
        "b": cfg.b,
        "L": cfg.L,
        "A": cfg.area,
        "I_g": cfg.I_g,
        "D0": cfg.D0,
        "m0": cfg.m0,
        "mu": cfg.mu,
        "ell": cfg.ell,

        # Adam epochs and L-BFGS iterations are not computationally
        # equivalent. This value is only a global plotting/bookkeeping
        # coordinate.
        "nominal_optimization_budget": int(
            cfg.adam_epochs
            + cfg.lbfgs_max_iter
        ),
    }

    with open(
        output_dir
        / "config.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            config_dict,
            f,
            indent=2,
        )

    # Research-grade startup checks
    verify_closed_form_section_resultants(
        cfg
    )

    validate_coefficient_profiles(
        cfg
    )

    verify_compact_vs_expanded_operator(
        cfg,
        device,
    )

    save_coefficient_profiles(
        cfg,
        output_dir,
    )

    # Fixed full-batch interior collocation points
    X_colloc = torch.linspace(
        cfg.x_min,
        cfg.x_max,
        cfg.n_colloc + 2,
        dtype=torch.float64,
        device=device,
    )[1:-1].reshape(
        -1,
        1,
    )

    # Fixed Gauss-Legendre quadrature
    X_quad, W_quad = gauss_legendre_tensors(
        n=cfg.n_quad,
        device=device,
        x_min=cfg.x_min,
        x_max=cfg.x_max,
    )

    # Dense evaluation grid
    X_test = torch.linspace(
        cfg.x_min,
        cfg.x_max,
        cfg.n_test,
        dtype=torch.float64,
        device=device,
    ).reshape(
        -1,
        1,
    )

    # Optional independent S-S BVP reference for snapshot visualization
    snapshot_reference = None
    snapshot_reference_x = None
    snapshot_reference_phi = None
    snapshot_reference_lambda = None

    if (
        cfg.save_epoch_plots
        and cfg.plot_every > 0
        and cfg.run_reference_bvp
    ):
        try:
            snapshot_reference = (
                solve_reference_bvp_ss(
                    cfg
                )
            )

            if snapshot_reference is not None:
                snapshot_reference_x = (
                    X_test
                    .detach()
                    .cpu()
                    .numpy()
                    .reshape(-1)
                )

                snapshot_reference_phi = (
                    snapshot_reference[
                        "solution"
                    ]
                    .sol(
                        snapshot_reference_x
                    )[0]
                    .reshape(-1)
                )

                snapshot_reference_lambda = float(
                    snapshot_reference[
                        "lambda"
                    ]
                )

                LOGGER.info(
                    "Live S-S snapshot reference prepared: "
                    "lambda_BVP = %.9f",
                    snapshot_reference_lambda,
                )

            else:
                LOGGER.warning(
                    "No independent S-S BVP reference is available "
                    "for snapshot visualization."
                )

        except Exception as exc:
            LOGGER.warning(
                "Failed to prepare independent S-S BVP reference "
                "for snapshot visualization: %s",
                exc,
            )

            snapshot_reference = None
            snapshot_reference_x = None
            snapshot_reference_phi = None
            snapshot_reference_lambda = None

    # PINN model
    model = EigenPINN(
        cfg
    ).to(
        device
    )

    # Verify the new hard S-S representation before any optimization.
    verify_ss_hard_constraint_structure(
        model=model,
        cfg=cfg,
        device=device,
    )

    nn_parameter_count = sum(
        parameter.numel()
        for parameter
        in model.nn_phi.parameters()
        if parameter.requires_grad
    )

    shape_parameter_count = (
        nn_parameter_count
        + model.slope_left.numel()
        + model.slope_right.numel()
    )

    total_parameter_count = sum(
        parameter.numel()
        for parameter
        in model.parameters()
        if parameter.requires_grad
    )

    LOGGER.info(
        "Trainable NN parameters: %d",
        nn_parameter_count,
    )

    LOGGER.info(
        "Trainable eigenfunction-shape parameters including "
        "two endpoint slopes: %d",
        shape_parameter_count,
    )

    LOGGER.info(
        "Total trainable parameters including endpoint slopes "
        "and lambda_raw: %d",
        total_parameter_count,
    )

    LOGGER.info(
        "Initial endpoint-slope parameters: "
        "slope_left=%.9f | slope_right=%.9f",
        float(
            model.slope_left
            .detach()
            .cpu()
        ),
        float(
            model.slope_right
            .detach()
            .cpu()
        ),
    )

    LOGGER.info(
        "No reference eigenvalue/eigenfunction is used during optimization."
    )

    # History
    history = make_history()

    # Globally best physics-informed state across BOTH optimizer phases
    best = {
        "loss": math.inf,

        "optimization_step": 0,
        "optimizer_phase": "",
        "phase_step": 0,

        "model_state": None,
        "checkpoint_state": "",

        "optimizer_nn_state": None,
        "optimizer_lambda_state": None,

        "scheduler_nn_state": None,
        "scheduler_lambda_state": None,

        "optimizer_lbfgs_state": None,
    }

    def save_best_checkpoint() -> None:
        """
        Save the currently recorded globally best physics-informed state.
        """
        torch.save(
            {
                "optimization_step": int(
                    best[
                        "optimization_step"
                    ]
                ),

                "optimizer_phase": str(
                    best[
                        "optimizer_phase"
                    ]
                ),

                "phase_step": int(
                    best[
                        "phase_step"
                    ]
                ),

                "best_loss": float(
                    best[
                        "loss"
                    ]
                ),

                "model_state_dict": (
                    best[
                        "model_state"
                    ]
                ),

                "optimizer_nn_state_dict": (
                    best[
                        "optimizer_nn_state"
                    ]
                ),

                "optimizer_lambda_state_dict": (
                    best[
                        "optimizer_lambda_state"
                    ]
                ),

                "scheduler_nn_state_dict": (
                    best[
                        "scheduler_nn_state"
                    ]
                ),

                "scheduler_lambda_state_dict": (
                    best[
                        "scheduler_lambda_state"
                    ]
                ),

                "optimizer_lbfgs_state_dict": (
                    best[
                        "optimizer_lbfgs_state"
                    ]
                ),

                "config": config_dict,

                "model_name": (
                    "Reduced Flexural "
                    "2D-FG EB-NSGT S-S"
                ),

                "boundary_condition": (
                    "Li-type simply-supported "
                    "fixed-curvature S-S"
                ),

                "checkpoint_state": str(
                    best[
                        "checkpoint_state"
                    ]
                ),

                "reference_used_in_training": False,
            },

            output_dir
            / "best_checkpoint.pt",
        )

    def maybe_update_best(
        current_loss: float,
        optimization_step: int,
        optimizer_phase: str,
        phase_step: int,
        checkpoint_state: str,
        optimizer_nn: Optional[
            torch.optim.Optimizer
        ] = None,
        optimizer_lambda: Optional[
            torch.optim.Optimizer
        ] = None,
        scheduler_nn=None,
        scheduler_lambda=None,
        optimizer_lbfgs: Optional[
            torch.optim.Optimizer
        ] = None,
    ) -> None:
        """
        Update the globally best state using only the current PINN objective.

        No BVP/reference quantity participates in this decision.
        """
        if current_loss >= best[
            "loss"
        ]:
            return

        best[
            "loss"
        ] = float(
            current_loss
        )

        best[
            "optimization_step"
        ] = int(
            optimization_step
        )

        best[
            "optimizer_phase"
        ] = str(
            optimizer_phase
        )

        best[
            "phase_step"
        ] = int(
            phase_step
        )

        best[
            "model_state"
        ] = cpu_state_dict(
            model
        )

        best[
            "checkpoint_state"
        ] = str(
            checkpoint_state
        )

        best[
            "optimizer_nn_state"
        ] = (
            None
            if optimizer_nn is None
            else cpu_optimizer_state_dict(
                optimizer_nn
            )
        )

        best[
            "optimizer_lambda_state"
        ] = (
            None
            if optimizer_lambda is None
            else cpu_optimizer_state_dict(
                optimizer_lambda
            )
        )

        best[
            "scheduler_nn_state"
        ] = (
            None
            if scheduler_nn is None
            else _to_cpu_recursive(
                scheduler_nn.state_dict()
            )
        )

        best[
            "scheduler_lambda_state"
        ] = (
            None
            if scheduler_lambda is None
            else _to_cpu_recursive(
                scheduler_lambda.state_dict()
            )
        )

        best[
            "optimizer_lbfgs_state"
        ] = (
            None
            if optimizer_lbfgs is None
            else cpu_optimizer_state_dict(
                optimizer_lbfgs
            )
        )

        save_best_checkpoint()

    # Stage I — Adam
    shape_parameters = [
        *list(
            model.nn_phi.parameters()
        ),
        model.slope_left,
        model.slope_right,
    ]

    optimizer_nn = torch.optim.Adam(
        shape_parameters,
        lr=cfg.lr_nn,
    )

    optimizer_lambda = torch.optim.Adam(
        [
            model.lambda_raw
        ],
        lr=cfg.lr_lambda,
    )

    milestones = list(
        cfg.lr_milestones
    )

    scheduler_nn = (
        torch.optim.lr_scheduler.MultiStepLR(
            optimizer_nn,
            milestones=milestones,
            gamma=cfg.lr_gamma,
        )
    )

    scheduler_lambda = (
        torch.optim.lr_scheduler.MultiStepLR(
            optimizer_lambda,
            milestones=milestones,
            gamma=cfg.lr_gamma,
        )
    )

    LOGGER.info(
        "Adam LR milestones: %s; gamma=%.4g",
        milestones,
        cfg.lr_gamma,
    )

    # Adam loop
    for epoch_idx in range(
        cfg.adam_epochs
    ):
        model.train()

        optimizer_nn.zero_grad(
            set_to_none=True
        )

        optimizer_lambda.zero_grad(
            set_to_none=True
        )

        # PRE-update Adam physics-informed state
        total_loss, comp = compute_loss(
            model,
            X_colloc,
            X_quad,
            W_quad,
            cfg,
            device,
        )

        current_loss = float(
            total_loss
            .detach()
            .cpu()
        )

        epoch_number = (
            epoch_idx
            + 1
        )

        optimization_step = (
            epoch_number
        )

        if not math.isfinite(
            current_loss
        ):
            raise FloatingPointError(
                "Non-finite total loss at "
                f"Adam epoch {epoch_number}. "
                "Inspect coefficient regularity, learning rates, "
                "r-weighted normalization, active endpoint moments, "
                "and high-order derivatives."
            )

        lr_nn_now = float(
            optimizer_nn
            .param_groups[
                0
            ][
                "lr"
            ]
        )

        lr_lambda_now = float(
            optimizer_lambda
            .param_groups[
                0
            ][
                "lr"
            ]
        )

        # Adam history
        append_history(
            history=history,

            comp=comp,

            optimization_step=(
                optimization_step
            ),

            optimizer_phase="Adam",

            phase_step=(
                epoch_number
            ),

            lr_nn=lr_nn_now,

            lr_lambda=lr_lambda_now,

            lr_lbfgs=math.nan,

            lbfgs_func_evals=0,
        )

        # Global best checkpoint
        maybe_update_best(
            current_loss=current_loss,

            optimization_step=(
                optimization_step
            ),

            optimizer_phase="Adam",

            phase_step=(
                epoch_number
            ),

            checkpoint_state=(
                "adam_pre_update"
            ),

            optimizer_nn=(
                optimizer_nn
            ),

            optimizer_lambda=(
                optimizer_lambda
            ),

            scheduler_nn=(
                scheduler_nn
            ),

            scheduler_lambda=(
                scheduler_lambda
            ),

            optimizer_lbfgs=None,
        )

        # Adam logging
        if (
            cfg.log_every > 0
            and (
                epoch_number == 1
                or (
                    epoch_number
                    % cfg.log_every
                    == 0
                )
                or (
                    epoch_number
                    == cfg.adam_epochs
                )
            )
        ):
            LOGGER.info(
                "[Adam] epoch %d/%d | "
                "total=%.6e | "
                "DE=%.6e | "
                "BC=%.6e | "
                "BC_raw=%.3e | "
                "Norm=%.6e | "
                "L2=%.3e | "
                "lambda=%.9f | "
                "int(r*phi^2)=%.8f | "
                "R_rms=%.3e | "
                "R_rel=%.3e | "
                "Mcore_rms=%.3e | "
                "Mbar_rms=%.3e | "
                "hard_BC_max=%.3e | "
                "lr=(%.2e, %.2e)",
                epoch_number,
                cfg.adam_epochs,

                current_loss,

                float(
                    comp[
                        "de"
                    ]
                    .detach()
                    .cpu()
                ),

                float(
                    comp[
                        "bc"
                    ]
                    .detach()
                    .cpu()
                ),

                float(
                    comp[
                        "bc_raw"
                    ]
                    .detach()
                    .cpu()
                ),

                float(
                    comp[
                        "norm"
                    ]
                    .detach()
                    .cpu()
                ),

                float(
                    comp[
                        "l2"
                    ]
                    .detach()
                    .cpu()
                ),

                float(
                    comp[
                        "lambda"
                    ]
                    .detach()
                    .cpu()
                ),

                float(
                    comp[
                        "mass_norm_integral"
                    ]
                    .detach()
                    .cpu()
                ),

                float(
                    comp[
                        "residual_rms"
                    ]
                    .detach()
                    .cpu()
                ),

                float(
                    comp[
                        "relative_residual_rms"
                    ]
                    .detach()
                    .cpu()
                ),

                float(
                    comp[
                        "moment_core_rms"
                    ]
                    .detach()
                    .cpu()
                ),

                float(
                    comp[
                        "physical_moment_rms"
                    ]
                    .detach()
                    .cpu()
                ),

                float(
                    comp[
                        "hard_bc_max_abs"
                    ]
                    .detach()
                    .cpu()
                ),

                lr_nn_now,
                lr_lambda_now,
            )

        # Adam snapshots
        if (
            cfg.save_epoch_plots
            and cfg.plot_every > 0
            and (
                epoch_number == 1
                or (
                    epoch_number
                    % cfg.plot_every
                    == 0
                )
                or (
                    epoch_number
                    == cfg.adam_epochs
                )
            )
        ):
            plot_snapshot(
                model=model,

                X_test=X_test,

                history=history,

                optimization_step=(
                    optimization_step
                ),

                output_path=(
                    plot_dir
                    / (
                        f"step_"
                        f"{optimization_step:06d}"
                        f".png"
                    )
                ),

                reference_x=(
                    snapshot_reference_x
                ),

                reference_phi=(
                    snapshot_reference_phi
                ),

                reference_lambda=(
                    snapshot_reference_lambda
                ),
            )

            # predict()/plot_snapshot() switches the model to eval().
            model.train()

            gc.collect()

        # Adam backpropagation
        total_loss.backward()

        # Gradient validity check
        for (
            name,
            parameter,
        ) in model.named_parameters():
            if parameter.grad is None:
                continue

            if not torch.isfinite(
                parameter.grad
            ).all().item():
                raise FloatingPointError(
                    "Non-finite gradient detected for parameter "
                    f"'{name}' at Adam epoch {epoch_number}."
                )

        # Optional gradient clipping — Adam only
        if (
            cfg.grad_clip_norm
            is not None
        ):
            torch.nn.utils.clip_grad_norm_(
                shape_parameters,
                cfg.grad_clip_norm,
            )

            torch.nn.utils.clip_grad_norm_(
                [
                    model.lambda_raw
                ],
                cfg.grad_clip_norm,
            )

        # Adam parameter and learning-rate updates
        optimizer_nn.step()

        optimizer_lambda.step()

        scheduler_nn.step()

        scheduler_lambda.step()

        del total_loss
        del comp

    # Exact POST-update Adam handoff state
    model.train()

    (
        adam_handoff_total,
        adam_handoff_comp,
    ) = compute_loss(
        model,
        X_colloc,
        X_quad,
        W_quad,
        cfg,
        device,
    )

    adam_handoff_loss = float(
        adam_handoff_total
        .detach()
        .cpu()
    )

    if not math.isfinite(
        adam_handoff_loss
    ):
        raise FloatingPointError(
            "Non-finite physics-informed loss at the "
            "Adam-to-L-BFGS handoff."
        )

    maybe_update_best(
        current_loss=(
            adam_handoff_loss
        ),

        optimization_step=(
            cfg.adam_epochs
        ),

        optimizer_phase="Adam",

        phase_step=(
            cfg.adam_epochs
        ),

        checkpoint_state=(
            "adam_post_update_handoff"
        ),

        optimizer_nn=(
            optimizer_nn
        ),

        optimizer_lambda=(
            optimizer_lambda
        ),

        scheduler_nn=(
            scheduler_nn
        ),

        scheduler_lambda=(
            scheduler_lambda
        ),

        optimizer_lbfgs=None,
    )

    torch.save(
        cpu_state_dict(
            model
        ),
        output_dir
        / "adam_handoff_state_dict.pt",
    )

    LOGGER.info(
        "Adam handoff state: "
        "total=%.6e | "
        "BC=%.6e | "
        "lambda=%.9f | "
        "R_rel=%.3e | "
        "Mcore_rms=%.3e | "
        "hard_BC_max=%.3e",
        adam_handoff_loss,

        float(
            adam_handoff_comp[
                "bc"
            ]
            .detach()
            .cpu()
        ),

        float(
            adam_handoff_comp[
                "lambda"
            ]
            .detach()
            .cpu()
        ),

        float(
            adam_handoff_comp[
                "relative_residual_rms"
            ]
            .detach()
            .cpu()
        ),

        float(
            adam_handoff_comp[
                "moment_core_rms"
            ]
            .detach()
            .cpu()
        ),

        float(
            adam_handoff_comp[
                "hard_bc_max_abs"
            ]
            .detach()
            .cpu()
        ),
    )

    del adam_handoff_total
    del adam_handoff_comp

    gc.collect()

    # Stage II — deterministic full-batch L-BFGS
    lbfgs_params = [
        parameter
        for parameter
        in model.parameters()
        if parameter.requires_grad
    ]

    if not lbfgs_params:
        raise RuntimeError(
            "No trainable parameters are available for L-BFGS."
        )

    initial_block = min(
        cfg.lbfgs_block_iter,
        cfg.lbfgs_max_iter,
    )

    optimizer_lbfgs = (
        torch.optim.LBFGS(
            lbfgs_params,

            lr=(
                cfg.lbfgs_lr
            ),

            max_iter=(
                initial_block
            ),

            # Deliberately generous evaluation budget so that a block is
            # not terminated merely by PyTorch's default max_eval limit.
            max_eval=max(
                25,
                5
                * initial_block,
            ),

            tolerance_grad=(
                cfg.lbfgs_tolerance_grad
            ),

            tolerance_change=(
                cfg.lbfgs_tolerance_change
            ),

            history_size=(
                cfg.lbfgs_history_size
            ),

            line_search_fn=(
                cfg.lbfgs_line_search_fn
            ),
        )
    )

    LOGGER.info(
        "Starting L-BFGS: "
        "lr=%.6g | "
        "max_iter=%d | "
        "block_iter=%d | "
        "history_size=%d | "
        "tol_grad=%.3e | "
        "tol_change=%.3e | "
        "line_search=%s",
        cfg.lbfgs_lr,
        cfg.lbfgs_max_iter,
        cfg.lbfgs_block_iter,
        cfg.lbfgs_history_size,
        cfg.lbfgs_tolerance_grad,
        cfg.lbfgs_tolerance_change,
        cfg.lbfgs_line_search_fn,
    )

    first_lbfgs_param = (
        lbfgs_params[
            0
        ]
    )

    lbfgs_iter = 0
    lbfgs_func_evals = 0

    # Persistent L-BFGS block loop
    while (
        lbfgs_iter
        < cfg.lbfgs_max_iter
    ):
        remaining = (
            cfg.lbfgs_max_iter
            - lbfgs_iter
        )

        requested_block = min(
            cfg.lbfgs_block_iter,
            remaining,
        )

        optimizer_lbfgs.param_groups[
            0
        ][
            "max_iter"
        ] = int(
            requested_block
        )

        optimizer_lbfgs.param_groups[
            0
        ][
            "max_eval"
        ] = max(
            25,
            5
            * int(
                requested_block
            ),
        )

        state_before = (
            optimizer_lbfgs.state[
                first_lbfgs_param
            ]
        )

        n_iter_before = int(
            state_before.get(
                "n_iter",
                0,
            )
        )

        func_evals_before = int(
            state_before.get(
                "func_evals",
                0,
            )
        )

        # Deterministic full-batch L-BFGS closure
        def closure() -> torch.Tensor:
            optimizer_lbfgs.zero_grad(
                set_to_none=True
            )

            model.train()

            loss, _ = compute_loss(
                model,
                X_colloc,
                X_quad,
                W_quad,
                cfg,
                device,
            )

            loss_value = float(
                loss
                .detach()
                .cpu()
            )

            if not math.isfinite(
                loss_value
            ):
                raise FloatingPointError(
                    "Non-finite total loss inside the L-BFGS closure."
                )

            loss.backward()

            # Gradient clipping is intentionally NOT used here. L-BFGS and
            # strong-Wolfe line search must see the actual objective gradient.
            for (
                name,
                parameter,
            ) in model.named_parameters():
                if parameter.grad is None:
                    continue

                if not torch.isfinite(
                    parameter.grad
                ).all().item():
                    raise FloatingPointError(
                        "Non-finite gradient detected for parameter "
                        f"'{name}' inside the L-BFGS closure."
                    )

            return loss

        # Execute one persistent L-BFGS block
        optimizer_lbfgs.step(
            closure
        )

        state_after = (
            optimizer_lbfgs.state[
                first_lbfgs_param
            ]
        )

        n_iter_after = int(
            state_after.get(
                "n_iter",
                n_iter_before,
            )
        )

        func_evals_after = int(
            state_after.get(
                "func_evals",
                func_evals_before,
            )
        )

        performed_iter = (
            n_iter_after
            - n_iter_before
        )

        performed_evals = (
            func_evals_after
            - func_evals_before
        )

        # PyTorch stores these counters cumulatively in the persistent state.
        lbfgs_iter = int(
            n_iter_after
        )

        lbfgs_func_evals = int(
            func_evals_after
        )

        # IMPORTANT:
        # If no new quasi-Newton iteration was accepted, do not append another
        # history entry at the same global optimization step.
        if performed_iter <= 0:
            LOGGER.info(
                "L-BFGS stopped early: no additional quasi-Newton "
                "iteration was accepted."
            )
            break

        # Recompute the accepted block-end state outside the closure.
        # Transient strong-Wolfe trial states are deliberately not recorded.
        model.train()

        (
            accepted_total,
            accepted_comp,
        ) = compute_loss(
            model,
            X_colloc,
            X_quad,
            W_quad,
            cfg,
            device,
        )

        accepted_loss = float(
            accepted_total
            .detach()
            .cpu()
        )

        if not math.isfinite(
            accepted_loss
        ):
            raise FloatingPointError(
                "Non-finite accepted-state loss after an L-BFGS block."
            )

        optimization_step = (
            cfg.adam_epochs
            + lbfgs_iter
        )

        # Accepted L-BFGS history
        append_history(
            history=history,

            comp=(
                accepted_comp
            ),

            optimization_step=(
                optimization_step
            ),

            optimizer_phase=(
                "L-BFGS"
            ),

            phase_step=(
                lbfgs_iter
            ),

            lr_nn=math.nan,

            lr_lambda=math.nan,

            lr_lbfgs=float(
                optimizer_lbfgs
                .param_groups[
                    0
                ][
                    "lr"
                ]
            ),

            lbfgs_func_evals=(
                lbfgs_func_evals
            ),
        )

        # Global best checkpoint
        maybe_update_best(
            current_loss=(
                accepted_loss
            ),

            optimization_step=(
                optimization_step
            ),

            optimizer_phase=(
                "L-BFGS"
            ),

            phase_step=(
                lbfgs_iter
            ),

            checkpoint_state=(
                "lbfgs_accepted_block_endpoint"
            ),

            optimizer_nn=None,

            optimizer_lambda=None,

            scheduler_nn=None,

            scheduler_lambda=None,

            optimizer_lbfgs=(
                optimizer_lbfgs
            ),
        )

        # L-BFGS logging
        LOGGER.info(
            "[L-BFGS] "
            "iter %d/%d | "
            "global_step=%d | "
            "block_iter=%d | "
            "block_evals=%d | "
            "total_evals=%d | "
            "total=%.6e | "
            "DE=%.6e | "
            "BC=%.6e | "
            "BC_raw=%.3e | "
            "lambda=%.9f | "
            "int(r*phi^2)=%.8f | "
            "R_rms=%.3e | "
            "R_rel=%.3e | "
            "Mcore_rms=%.3e | "
            "Mbar_rms=%.3e | "
            "hard_BC_max=%.3e",
            lbfgs_iter,
            cfg.lbfgs_max_iter,

            optimization_step,

            performed_iter,
            performed_evals,
            lbfgs_func_evals,

            accepted_loss,

            float(
                accepted_comp[
                    "de"
                ]
                .detach()
                .cpu()
            ),

            float(
                accepted_comp[
                    "bc"
                ]
                .detach()
                .cpu()
            ),

            float(
                accepted_comp[
                    "bc_raw"
                ]
                .detach()
                .cpu()
            ),

            float(
                accepted_comp[
                    "lambda"
                ]
                .detach()
                .cpu()
            ),

            float(
                accepted_comp[
                    "mass_norm_integral"
                ]
                .detach()
                .cpu()
            ),

            float(
                accepted_comp[
                    "residual_rms"
                ]
                .detach()
                .cpu()
            ),

            float(
                accepted_comp[
                    "relative_residual_rms"
                ]
                .detach()
                .cpu()
            ),

            float(
                accepted_comp[
                    "moment_core_rms"
                ]
                .detach()
                .cpu()
            ),

            float(
                accepted_comp[
                    "physical_moment_rms"
                ]
                .detach()
                .cpu()
            ),

            float(
                accepted_comp[
                    "hard_bc_max_abs"
                ]
                .detach()
                .cpu()
            ),
        )

        # L-BFGS snapshots
        crossed_plot_boundary = (
            cfg.save_epoch_plots
            and cfg.plot_every > 0
            and (
                (
                    lbfgs_iter
                    // cfg.plot_every
                )
                >
                (
                    n_iter_before
                    // cfg.plot_every
                )
                or (
                    lbfgs_iter
                    >= cfg.lbfgs_max_iter
                )
                or (
                    performed_iter
                    < requested_block
                )
            )
        )

        if crossed_plot_boundary:
            plot_snapshot(
                model=model,

                X_test=X_test,

                history=history,

                optimization_step=(
                    optimization_step
                ),

                output_path=(
                    plot_dir
                    / (
                        f"step_"
                        f"{optimization_step:06d}"
                        f".png"
                    )
                ),

                reference_x=(
                    snapshot_reference_x
                ),

                reference_phi=(
                    snapshot_reference_phi
                ),

                reference_lambda=(
                    snapshot_reference_lambda
                ),
            )

            model.train()

            gc.collect()

        del accepted_total
        del accepted_comp

        gc.collect()

        # Internal L-BFGS early termination
        if (
            performed_iter
            < requested_block
        ):
            LOGGER.info(
                "L-BFGS stopped early after %d cumulative iterations; "
                "the internal stopping criterion completed %d of %d "
                "requested iterations in the final block.",
                lbfgs_iter,
                performed_iter,
                requested_block,
            )
            break

    # Restore globally best physics-informed state across BOTH phases
    if (
        best[
            "model_state"
        ]
        is None
    ):
        raise RuntimeError(
            "No finite best model state was recorded."
        )

    model.load_state_dict(
        best[
            "model_state"
        ]
    )

    model.to(
        device
    )

    model.train()

    (
        best_total_tensor,
        best_comp_tensor,
    ) = compute_loss(
        model,
        X_colloc,
        X_quad,
        W_quad,
        cfg,
        device,
    )

    best_total_value = float(
        best_total_tensor
        .detach()
        .cpu()
    )

    best_comp = {
        key: float(
            value
            .detach()
            .cpu()
        )
        for (
            key,
            value,
        )
        in best_comp_tensor.items()
    }

    if not math.isclose(
        best_total_value,
        float(
            best[
                "loss"
            ]
        ),
        rel_tol=1.0e-9,
        abs_tol=1.0e-12,
    ):
        LOGGER.warning(
            "Restored best-loss check differs from recorded value: "
            "recorded=%.12e, recomputed=%.12e",
            float(
                best[
                    "loss"
                ]
            ),
            best_total_value,
        )

    best_step = int(
        best[
            "optimization_step"
        ]
    )

    best_phase = str(
        best[
            "optimizer_phase"
        ]
    )

    best_phase_step = int(
        best[
            "phase_step"
        ]
    )

    del best_total_tensor
    del best_comp_tensor

    gc.collect()

    # Best eigenvalue and physical frequency
    best_lambda = float(
        model.lambda_
        .detach()
        .cpu()
    )

    omega = lambda_to_omega(
        best_lambda,
        cfg,
    )

    frequency_hz = (
        lambda_to_frequency_hz(
            best_lambda,
            cfg,
        )
    )

    LOGGER.info(
        "Training complete."
    )

    LOGGER.info(
        "Best optimization step: %d | phase=%s | phase_step=%d",
        best_step,
        best_phase,
        best_phase_step,
    )

    LOGGER.info(
        "Adam epochs completed: %d",
        cfg.adam_epochs,
    )

    LOGGER.info(
        "L-BFGS iterations completed: %d/%d | function evaluations=%d",
        lbfgs_iter,
        cfg.lbfgs_max_iter,
        lbfgs_func_evals,
    )

    LOGGER.info(
        "Best total loss: %.12e",
        best_total_value,
    )

    LOGGER.info(
        "Best lambda: %.12f",
        best_lambda,
    )

    LOGGER.info(
        "Physical omega: %.12e rad/s",
        omega,
    )

    LOGGER.info(
        "Physical frequency: %.12e Hz",
        frequency_hz,
    )

    # Final S-S endpoint diagnostics
    LOGGER.info(
        "Best S-S endpoint diagnostics | "
        "X=0: Phi=%.3e, Phi'=%.3e, Phi''=%.3e, "
        "Mcore=%.3e, Mbar=%.3e | "
        "X=1: Phi=%.3e, Phi'=%.3e, Phi''=%.3e, "
        "Mcore=%.3e, Mbar=%.3e",
        best_comp[
            "phi_0"
        ],
        best_comp[
            "dphi_0"
        ],
        best_comp[
            "ddphi_0"
        ],
        best_comp[
            "moment_core_0"
        ],
        best_comp[
            "moment_0"
        ],
        best_comp[
            "phi_1"
        ],
        best_comp[
            "dphi_1"
        ],
        best_comp[
            "ddphi_1"
        ],
        best_comp[
            "moment_core_1"
        ],
        best_comp[
            "moment_1"
        ],
    )

    LOGGER.info(
        "Best S-S aggregate BC diagnostics | "
        "hard_BC_max=%.3e | "
        "Mcore_rms=%.3e | "
        "Mbar_rms=%.3e | "
        "weighted_BC_loss=%.3e",
        best_comp[
            "hard_bc_max_abs"
        ],
        best_comp[
            "moment_core_rms"
        ],
        best_comp[
            "physical_moment_rms"
        ],
        best_comp[
            "bc"
        ],
    )

    LOGGER.info(
        "Best raw endpoint-slope parameters | "
        "slope_left=%.9e | slope_right=%.9e",
        float(
            model.slope_left
            .detach()
            .cpu()
        ),
        float(
            model.slope_right
            .detach()
            .cpu()
        ),
    )

    # Save synchronized globally best model/checkpoint
    torch.save(
        cpu_state_dict(
            model
        ),
        output_dir
        / "best_state_dict.pt",
    )

    torch.save(
        {
            "best_optimization_step": (
                best_step
            ),

            "best_optimizer_phase": (
                best_phase
            ),

            "best_phase_step": (
                best_phase_step
            ),

            "best_loss": (
                best_total_value
            ),

            "lambda": (
                best_lambda
            ),

            "omega_rad_s": (
                omega
            ),

            "frequency_hz": (
                frequency_hz
            ),

            "raw_slope_left": float(
                model.slope_left
                .detach()
                .cpu()
            ),

            "raw_slope_right": float(
                model.slope_right
                .detach()
                .cpu()
            ),

            "model_state_dict": (
                cpu_state_dict(
                    model
                )
            ),

            "optimizer_nn_state_dict_at_best": (
                best[
                    "optimizer_nn_state"
                ]
            ),

            "optimizer_lambda_state_dict_at_best": (
                best[
                    "optimizer_lambda_state"
                ]
            ),

            "scheduler_nn_state_dict_at_best": (
                best[
                    "scheduler_nn_state"
                ]
            ),

            "scheduler_lambda_state_dict_at_best": (
                best[
                    "scheduler_lambda_state"
                ]
            ),

            "optimizer_lbfgs_state_dict_at_best": (
                best[
                    "optimizer_lbfgs_state"
                ]
            ),

            "checkpoint_state": str(
                best[
                    "checkpoint_state"
                ]
            ),

            "adam_epochs_completed": int(
                cfg.adam_epochs
            ),

            "lbfgs_iterations_completed": int(
                lbfgs_iter
            ),

            "lbfgs_function_evaluations": int(
                lbfgs_func_evals
            ),

            "config": (
                config_dict
            ),

            "model_name": (
                "Reduced Flexural "
                "2D-FG EB-NSGT S-S"
            ),

            "boundary_condition": (
                "Li-type simply-supported "
                "fixed-curvature S-S"
            ),

            "reference_used_in_training": False,
        },

        output_dir
        / "best_full.pt",
    )

    # Save complete synchronized training history
    save_history_csv(
        history,
        output_dir,
    )

    # Final PINN post-processing
    model.eval()

    X_np = (
        X_test
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )

    X_quad_np = (
        X_quad
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )

    W_quad_np = (
        W_quad
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )

    sec_test = section_resultants_numpy(
        X_np,
        cfg,
    )

    r_test = (
        sec_test[
            "r"
        ]
    )

    # Raw PINN eigenfunction
    phi_pinn_raw = predict(
        model,
        X_test,
    )

    pinn_mass_norm_quad = float(
        best_comp[
            "raw_mass_norm_integral"
        ]
    )

    if (
        not math.isfinite(
            pinn_mass_norm_quad
        )
        or (
            pinn_mass_norm_quad
            <= 1.0e-300
        )
    ):
        raise RuntimeError(
            "Invalid PINN r-weighted normalization integral "
            "during post-processing."
        )

    # Explicitly normalized exported PINN mode.
    phi_pinn = (
        phi_pinn_raw
        / math.sqrt(
            pinn_mass_norm_quad
        )
    )

    pinn_mass_norm_test = float(
        np.trapezoid(
            r_test
            * phi_pinn**2,
            X_np,
        )
    )

    phi_pinn_plot = (
        maxabs_normalize_mode(
            phi_pinn
        )
    )

    pd.DataFrame(
        {
            "X": X_np,

            "Phi_PINN_raw": (
                phi_pinn_raw
            ),

            "Phi_PINN_r_weighted_normalized": (
                phi_pinn
            ),

            "Phi_PINN_maxabs_normalized": (
                phi_pinn_plot
            ),

            "d": (
                sec_test[
                    "d"
                ]
            ),

            "r": (
                sec_test[
                    "r"
                ]
            ),
        }
    ).to_csv(
        output_dir
        / "best_pinn_eigenfunction.csv",
        index=False,
    )

    # Independent BVP validation
    if (
        snapshot_reference
        is not None
    ):
        ref = (
            snapshot_reference
        )

    else:
        ref = solve_reference_bvp_ss(
            cfg
        )

    validation = {
        "boundary_condition": (
            "Li-type simply-supported fixed-curvature S-S"
        ),

        "best_optimization_step": int(
            best_step
        ),

        "best_optimizer_phase": str(
            best_phase
        ),

        "best_phase_step": int(
            best_phase_step
        ),

        "adam_epochs_completed": int(
            cfg.adam_epochs
        ),

        "lbfgs_iterations_completed": int(
            lbfgs_iter
        ),

        "lbfgs_function_evaluations": int(
            lbfgs_func_evals
        ),

        "best_total_loss": float(
            best_total_value
        ),

        "best_de_loss": float(
            best_comp[
                "de"
            ]
        ),

        "best_bc_loss": float(
            best_comp[
                "bc"
            ]
        ),

        "best_bc_raw": float(
            best_comp[
                "bc_raw"
            ]
        ),

        "lambda_pinn": float(
            best_lambda
        ),

        "omega_pinn_rad_s": float(
            omega
        ),

        "frequency_pinn_hz": float(
            frequency_hz
        ),

        "normalized_r_weighted_integral": float(
            best_comp[
                "mass_norm_integral"
            ]
        ),

        "raw_pinn_r_weighted_integral": float(
            best_comp[
                "raw_mass_norm_integral"
            ]
        ),

        "pinn_r_weighted_norm_postprocessed": float(
            pinn_mass_norm_test
        ),

        "pde_residual_rms": float(
            best_comp[
                "residual_rms"
            ]
        ),

        "operator_scale_rms": float(
            best_comp[
                "operator_scale_rms"
            ]
        ),

        "relative_pde_residual_rms": float(
            best_comp[
                "relative_residual_rms"
            ]
        ),

        "max_abs_pde_residual": float(
            best_comp[
                "max_abs_residual"
            ]
        ),

        # Exact hard S-S conditions
        "phi_0": float(
            best_comp[
                "phi_0"
            ]
        ),

        "ddphi_0": float(
            best_comp[
                "ddphi_0"
            ]
        ),

        "phi_1": float(
            best_comp[
                "phi_1"
            ]
        ),

        "ddphi_1": float(
            best_comp[
                "ddphi_1"
            ]
        ),

        "hard_bc_max_abs": float(
            best_comp[
                "hard_bc_max_abs"
            ]
        ),

        # Free endpoint rotations — diagnostics only
        "dphi_0_free": float(
            best_comp[
                "dphi_0"
            ]
        ),

        "dphi_1_free": float(
            best_comp[
                "dphi_1"
            ]
        ),

        # Raw trainable slope parameters before differentiable normalization.
        "raw_slope_left": float(
            model.slope_left
            .detach()
            .cpu()
        ),

        "raw_slope_right": float(
            model.slope_right
            .detach()
            .cpu()
        ),

        # Active S-S moment conditions
        "moment_core_0": float(
            best_comp[
                "moment_core_0"
            ]
        ),

        "moment_core_1": float(
            best_comp[
                "moment_core_1"
            ]
        ),

        "moment_core_rms": float(
            best_comp[
                "moment_core_rms"
            ]
        ),

        "moment_0": float(
            best_comp[
                "moment_0"
            ]
        ),

        "moment_1": float(
            best_comp[
                "moment_1"
            ]
        ),

        "physical_moment_rms": float(
            best_comp[
                "physical_moment_rms"
            ]
        ),

        "reference_used_in_training": False,
    }

    # Independent BVP reference available
    if ref is not None:
        sol = (
            ref[
                "solution"
            ]
        )

        lambda_ref = float(
            ref[
                "lambda"
            ]
        )

        # Independent BVP normalization using the same Gauss-Legendre rule
        # used for the exported PINN normalization.
        phi_ref_quad_raw = (
            sol.sol(
                X_quad_np
            )[
                0
            ]
        )

        r_quad_np = (
            section_resultants_numpy(
                X_quad_np,
                cfg,
            )[
                "r"
            ]
        )

        ref_mass_norm_quad = float(
            np.sum(
                W_quad_np
                * r_quad_np
                * phi_ref_quad_raw**2
            )
        )

        if (
            not math.isfinite(
                ref_mass_norm_quad
            )
            or (
                ref_mass_norm_quad
                <= 1.0e-300
            )
        ):
            raise RuntimeError(
                "Invalid independent S-S BVP r-weighted "
                "normalization integral."
            )

        phi_ref_raw = (
            sol.sol(
                X_np
            )[
                0
            ]
        )

        phi_ref = (
            phi_ref_raw
            / math.sqrt(
                ref_mass_norm_quad
            )
        )

        # Resolve arbitrary eigenfunction sign
        inner = float(
            np.trapezoid(
                r_test
                * phi_pinn
                * phi_ref,
                X_np,
            )
        )

        reference_sign = (
            1.0
        )

        if inner < 0.0:
            reference_sign = (
                -1.0
            )

            phi_ref = (
                -phi_ref
            )

        phi_ref_raw_aligned = (
            reference_sign
            * phi_ref_raw
        )

        # r-weighted mode-shape metrics
        metrics = weighted_mode_metrics(
            X_np,
            phi_pinn,
            phi_ref,
            cfg,
        )

        # Eigenvalue and physical-frequency errors
        lambda_rel_err = (
            abs(
                best_lambda
                - lambda_ref
            )
            / lambda_ref
        )

        omega_ref = (
            lambda_to_omega(
                lambda_ref,
                cfg,
            )
        )

        frequency_ref_hz = (
            lambda_to_frequency_hz(
                lambda_ref,
                cfg,
            )
        )

        frequency_rel_err = (
            abs(
                frequency_hz
                - frequency_ref_hz
            )
            / frequency_ref_hz
        )

        ref_mass_norm_test = float(
            np.trapezoid(
                r_test
                * phi_ref**2,
                X_np,
            )
        )

        validation.update(
            {
                "lambda_reference_bvp": float(
                    lambda_ref
                ),

                "lambda_relative_error": float(
                    lambda_rel_err
                ),

                "lambda_relative_error_percent": float(
                    100.0
                    * lambda_rel_err
                ),

                "omega_reference_bvp_rad_s": float(
                    omega_ref
                ),

                "frequency_reference_bvp_hz": float(
                    frequency_ref_hz
                ),

                "frequency_relative_error": float(
                    frequency_rel_err
                ),

                "frequency_relative_error_percent": float(
                    100.0
                    * frequency_rel_err
                ),

                **metrics,

                "reference_bvp_max_rms_residual": float(
                    ref.get(
                        "max_rms_residual",
                        np.max(
                            sol.rms_residuals
                        ),
                    )
                ),

                "reference_bvp_mesh_nodes": int(
                    ref.get(
                        "mesh_nodes",
                        sol.x.size,
                    )
                ),

                "reference_bvp_raw_r_weighted_norm_gauss": float(
                    ref_mass_norm_quad
                ),

                "reference_bvp_r_weighted_norm_postprocessed": float(
                    ref_mass_norm_test
                ),
            }
        )

        # Metadata returned directly by the independent S-S BVP solver
        if "mode_number" in ref:
            validation[
                "reference_bvp_mode_number_candidate"
            ] = int(
                ref[
                    "mode_number"
                ]
            )

        if "interior_nodes" in ref:
            validation[
                "reference_bvp_interior_nodes"
            ] = int(
                ref[
                    "interior_nodes"
                ]
            )

            validation[
                "reference_bvp_zero_node_fundamental_candidate"
            ] = bool(
                ref[
                    "interior_nodes"
                ]
                == 0
            )

        if "initial_guess" in ref:
            validation[
                "reference_bvp_initial_lambda_guess"
            ] = float(
                ref[
                    "initial_guess"
                ]
            )

        if (
            "r_weighted_norm_integral"
            in ref
        ):
            validation[
                "reference_bvp_solver_r_weighted_norm_integral"
            ] = float(
                ref[
                    "r_weighted_norm_integral"
                ]
            )

        if "moment_0" in ref:
            validation[
                "reference_bvp_moment_0"
            ] = float(
                ref[
                    "moment_0"
                ]
            )

        if "moment_1" in ref:
            validation[
                "reference_bvp_moment_1"
            ] = float(
                ref[
                    "moment_1"
                ]
            )

        LOGGER.info(
            "Post-training S-S validation: "
            "lambda_ref=%.12f | "
            "lambda err=%.6f%% | "
            "r-weighted MAC=%.10f | "
            "mode rel.L2=%.6e | "
            "frequency err=%.6f%%",
            lambda_ref,

            100.0
            * lambda_rel_err,

            metrics[
                "r_weighted_MAC"
            ],

            metrics[
                "relative_L2_mode_error"
            ],

            100.0
            * frequency_rel_err,
        )

        # Consistent visualization orientation
        pinn_plot_scale = float(
            np.max(
                np.abs(
                    phi_pinn
                )
            )
        )

        ref_plot_scale = float(
            np.max(
                np.abs(
                    phi_ref
                )
            )
        )

        if (
            pinn_plot_scale
            <= 1.0e-14
            or
            ref_plot_scale
            <= 1.0e-14
        ):
            raise RuntimeError(
                "Near-zero mode amplitude encountered during "
                "S-S comparison plotting."
            )

        anchor_idx = int(
            np.argmax(
                np.abs(
                    phi_pinn
                )
            )
        )

        common_plot_sign = (
            -1.0
            if (
                phi_pinn[
                    anchor_idx
                ]
                < 0.0
            )
            else 1.0
        )

        phi_pinn_plot_compare = (
            common_plot_sign
            * phi_pinn
            / pinn_plot_scale
        )

        phi_ref_plot = (
            common_plot_sign
            * phi_ref
            / ref_plot_scale
        )

        # Save PINN-vs-BVP mode shapes
        pd.DataFrame(
            {
                "X": X_np,

                "Phi_PINN_raw": (
                    phi_pinn_raw
                ),

                "Phi_BVP_raw_sign_aligned": (
                    phi_ref_raw_aligned
                ),

                "Phi_PINN_r_weighted_normalized": (
                    phi_pinn
                ),

                "Phi_BVP_r_weighted_normalized": (
                    phi_ref
                ),

                "Phi_PINN_maxabs_normalized": (
                    phi_pinn_plot_compare
                ),

                "Phi_BVP_maxabs_normalized": (
                    phi_ref_plot
                ),
            }
        ).to_csv(
            output_dir
            / "pinn_vs_bvp_eigenfunction.csv",
            index=False,
        )

        # Final publication-style comparison
        plot_final_mode_comparison(
            X=X_np,

            phi_pinn=(
                phi_pinn
            ),

            lambda_pinn=(
                best_lambda
            ),

            optimization_step=(
                best_step
            ),

            output_path=(
                output_dir
                / "best_eigenfunction_vs_bvp.png"
            ),

            phi_ref=(
                phi_ref
            ),

            lambda_ref=(
                lambda_ref
            ),
        )

    # No BVP reference available
    else:
        plot_final_mode_comparison(
            X=X_np,

            phi_pinn=(
                phi_pinn
            ),

            lambda_pinn=(
                best_lambda
            ),

            optimization_step=(
                best_step
            ),

            output_path=(
                output_dir
                / "best_eigenfunction.png"
            ),

            phi_ref=None,

            lambda_ref=None,
        )

    # Save validation summary
    with open(
        output_dir
        / "validation_summary.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            validation,
            f,
            indent=2,
        )

    # Save globally best optimization-state metadata
    with open(
        output_dir
        / "best_optimization_step.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "best_optimization_step": int(
                    best_step
                ),

                "best_optimizer_phase": str(
                    best_phase
                ),

                "best_phase_step": int(
                    best_phase_step
                ),

                "adam_epochs_completed": int(
                    cfg.adam_epochs
                ),

                "lbfgs_iterations_completed": int(
                    lbfgs_iter
                ),

                "lbfgs_function_evaluations": int(
                    lbfgs_func_evals
                ),
            },
            f,
            indent=2,
        )

    # Return results to Jupyter / Python
    return {
        "model": model,

        "history": history,

        "best_optimization_step": int(
            best_step
        ),

        "best_optimizer_phase": str(
            best_phase
        ),

        "best_phase_step": int(
            best_phase_step
        ),

        "adam_epochs_completed": int(
            cfg.adam_epochs
        ),

        "lbfgs_iterations_completed": int(
            lbfgs_iter
        ),

        "lbfgs_function_evaluations": int(
            lbfgs_func_evals
        ),

        "best_loss": float(
            best_total_value
        ),

        "best_lambda": float(
            best_lambda
        ),

        "omega_rad_s": float(
            omega
        ),

        "frequency_hz": float(
            frequency_hz
        ),

        "validation": validation,

        "device": str(
            device
        ),
    }

<div class="alert alert-danger" role="alert"> 
🔎 Main

In [ ]:
# Jupyter Notebook configuration and execution — S-S hybrid protocol

cfg = Config(
    # 2D-FG material gradation
    k=1.0,
    beta=1.0,

    # NSGT
    tau=0.05,
    zeta=0.05,

    # Geometry
    L_over_h=30.0,

    # PINN discretization
    n_colloc=128,
    n_quad=128,
    n_test=401,

    x_min=0.0,
    x_max=1.0,

    # Neural network
    hidden_1=128,
    hidden_2=64,
    hidden_3=32,
    use_bias=True,

    # Hybrid optimization protocol
    # Stage I  : Adam   — 20,000 epochs
    # Stage II : L-BFGS — up to 10,000 quasi-Newton iterations

    # Stage I — Adam
    adam_epochs=20_000,

    lr_nn=2.0e-3,
    lr_lambda=1.0e-3,

    lambda_init=100.0,

    lr_milestones=(
        3_000,
        4_250,
        15_000,
    ),

    lr_gamma=0.20,

    # Stage II — persistent full-batch L-BFGS
    lbfgs_max_iter=10_000,
    lbfgs_block_iter=100,

    lbfgs_lr=1.0,
    lbfgs_history_size=100,

    lbfgs_tolerance_grad=1.0e-9,
    lbfgs_tolerance_change=1.0e-12,

    lbfgs_line_search_fn="strong_wolfe",

    # Physics-informed objective
    alpha_de=1.0,
    alpha_bc=5.0,
    alpha_norm=0.0,

    l2_coeff=1.0e-6,
    apply_l2=False,

    grad_clip_norm=None,

    # Reproducibility
    seed=42,
    deterministic_torch=True,

    # Logging / output
    log_every=50,
    plot_every=500,
    save_epoch_plots=True,

    output_dir=(
        "results_2dfg_eb_nsgt_ss_"
        "adam20k_lbfgs10k_seed42"
    ),

    device="auto",

    # Independent S-S BVP validation
    run_reference_bvp=True,

    reference_lambda_guess=100.0,
    reference_tol=1.0e-7,
    reference_max_nodes=20_000,
)


# Run S-S hybrid training

results = train(
    cfg
)


# Compact final S-S summary

print(
    "\n"
    + "=" * 82
)

print(
    "FINAL S-S PINN RESULTS — ADAM 20k + L-BFGS 10k"
)

print(
    "=" * 82
)

print(
    f"Device                    : "
    f"{results['device']}"
)

print(
    f"Adam epochs completed     : "
    f"{results['adam_epochs_completed']}"
)

print(
    f"L-BFGS iterations         : "
    f"{results['lbfgs_iterations_completed']}"
)

print(
    f"L-BFGS function evals     : "
    f"{results['lbfgs_function_evaluations']}"
)

print(
    f"Best optimization step    : "
    f"{results['best_optimization_step']}"
)

print(
    f"Best optimizer phase      : "
    f"{results['best_optimizer_phase']}"
)

print(
    f"Best phase step           : "
    f"{results['best_phase_step']}"
)

print(
    f"Best total loss           : "
    f"{results['best_loss']:.12e}"
)

print(
    f"Best lambda               : "
    f"{results['best_lambda']:.12f}"
)

print(
    f"Omega [rad/s]             : "
    f"{results['omega_rad_s']:.12e}"
)

print(
    f"Frequency [Hz]            : "
    f"{results['frequency_hz']:.12e}"
)


validation = results.get(
    "validation",
    {},
)


# Best physics-informed loss / PDE diagnostics

print(
    "-" * 82
)

print(
    "Best physics-informed diagnostics"
)

if "best_de_loss" in validation:
    print(
        f"DE loss                   : "
        f"{validation['best_de_loss']:.12e}"
    )

if "best_bc_loss" in validation:
    print(
        f"Weighted BC loss          : "
        f"{validation['best_bc_loss']:.12e}"
    )

if "best_bc_raw" in validation:
    print(
        f"Raw BC loss               : "
        f"{validation['best_bc_raw']:.12e}"
    )

if "normalized_r_weighted_integral" in validation:
    print(
        f"int(r*Phi^2) normalized   : "
        f"{validation['normalized_r_weighted_integral']:.12f}"
    )

if "raw_pinn_r_weighted_integral" in validation:
    print(
        f"Raw int(r*Phi^2)          : "
        f"{validation['raw_pinn_r_weighted_integral']:.12e}"
    )

if "pinn_r_weighted_norm_postprocessed" in validation:
    print(
        f"Postprocessed r-norm      : "
        f"{validation['pinn_r_weighted_norm_postprocessed']:.12f}"
    )

if "pde_residual_rms" in validation:
    print(
        f"PDE residual RMS          : "
        f"{validation['pde_residual_rms']:.6e}"
    )

if "operator_scale_rms" in validation:
    print(
        f"Operator scale RMS        : "
        f"{validation['operator_scale_rms']:.6e}"
    )

if "relative_pde_residual_rms" in validation:
    print(
        f"Relative PDE residual RMS : "
        f"{validation['relative_pde_residual_rms']:.6e}"
    )

if "max_abs_pde_residual" in validation:
    print(
        f"Max |PDE residual|        : "
        f"{validation['max_abs_pde_residual']:.6e}"
    )


# S-S endpoint diagnostics

print(
    "-" * 82
)

print(
    "S-S endpoint diagnostics"
)

if "phi_0" in validation:
    print(
        f"X=0 | Phi                : "
        f"{validation['phi_0']:.6e}"
    )

if "ddphi_0" in validation:
    print(
        f"X=0 | Phi''              : "
        f"{validation['ddphi_0']:.6e}"
    )

if "dphi_0_free" in validation:
    print(
        f"X=0 | Phi' free          : "
        f"{validation['dphi_0_free']:.6e}"
    )

if "moment_core_0" in validation:
    print(
        f"X=0 | Mcore              : "
        f"{validation['moment_core_0']:.6e}"
    )

if "moment_0" in validation:
    print(
        f"X=0 | Mbar               : "
        f"{validation['moment_0']:.6e}"
    )

if "phi_1" in validation:
    print(
        f"X=1 | Phi                : "
        f"{validation['phi_1']:.6e}"
    )

if "ddphi_1" in validation:
    print(
        f"X=1 | Phi''              : "
        f"{validation['ddphi_1']:.6e}"
    )

if "dphi_1_free" in validation:
    print(
        f"X=1 | Phi' free          : "
        f"{validation['dphi_1_free']:.6e}"
    )

if "moment_core_1" in validation:
    print(
        f"X=1 | Mcore              : "
        f"{validation['moment_core_1']:.6e}"
    )

if "moment_1" in validation:
    print(
        f"X=1 | Mbar               : "
        f"{validation['moment_1']:.6e}"
    )

if "hard_bc_max_abs" in validation:
    print(
        f"Hard BC max abs          : "
        f"{validation['hard_bc_max_abs']:.6e}"
    )

if "moment_core_rms" in validation:
    print(
        f"Mcore RMS                : "
        f"{validation['moment_core_rms']:.6e}"
    )

if "physical_moment_rms" in validation:
    print(
        f"Mbar RMS                 : "
        f"{validation['physical_moment_rms']:.6e}"
    )

if "raw_slope_left" in validation:
    print(
        f"Raw slope parameter left : "
        f"{validation['raw_slope_left']:.6e}"
    )

if "raw_slope_right" in validation:
    print(
        f"Raw slope parameter right: "
        f"{validation['raw_slope_right']:.6e}"
    )


# Independent S-S BVP validation

if "lambda_reference_bvp" in validation:
    print(
        "-" * 82
    )

    print(
        "Independent S-S BVP validation"
    )

    print(
        f"Reference lambda          : "
        f"{validation['lambda_reference_bvp']:.12f}"
    )

    print(
        f"Lambda error [%]          : "
        f"{validation['lambda_relative_error_percent']:.6f}"
    )

    if "frequency_relative_error_percent" in validation:
        print(
            f"Frequency error [%]       : "
            f"{validation['frequency_relative_error_percent']:.6f}"
        )

    if "r_weighted_MAC" in validation:
        print(
            f"r-weighted MAC            : "
            f"{validation['r_weighted_MAC']:.10f}"
        )

    if "relative_L2_mode_error" in validation:
        print(
            f"Mode relative L2 error    : "
            f"{validation['relative_L2_mode_error']:.6e}"
        )

    if "reference_bvp_interior_nodes" in validation:
        print(
            f"BVP interior nodes        : "
            f"{validation['reference_bvp_interior_nodes']}"
        )

    if "reference_bvp_initial_lambda_guess" in validation:
        print(
            f"BVP selected init. guess  : "
            f"{validation['reference_bvp_initial_lambda_guess']:.6f}"
        )

    if "reference_bvp_mesh_nodes" in validation:
        print(
            f"BVP mesh nodes            : "
            f"{validation['reference_bvp_mesh_nodes']}"
        )

    if "reference_bvp_max_rms_residual" in validation:
        print(
            f"BVP max RMS residual      : "
            f"{validation['reference_bvp_max_rms_residual']:.6e}"
        )

    if (
        "reference_bvp_solver_r_weighted_norm_integral"
        in validation
    ):
        print(
            f"BVP r-weighted norm       : "
            f"{validation['reference_bvp_solver_r_weighted_norm_integral']:.12f}"
        )

    if "reference_bvp_moment_0" in validation:
        print(
            f"BVP Mbar(0)               : "
            f"{validation['reference_bvp_moment_0']:.6e}"
        )

    if "reference_bvp_moment_1" in validation:
        print(
            f"BVP Mbar(1)               : "
            f"{validation['reference_bvp_moment_1']:.6e}"
        )


print(
    "=" * 82
)

2026-09-04 00:33:38,463 - INFO - Using device: cpu
2026-09-04 00:33:38,463 - INFO - Default dtype: torch.float64
2026-09-04 00:33:38,471 - INFO - Model: Reduced Flexural 2D-FG Euler-Bernoulli-NSGT, Li-type S-S fixed-curvature BC
2026-09-04 00:33:38,472 - INFO - k=1, beta=1, tau=0.05, zeta=0.05, L/h=30
2026-09-04 00:33:38,483 - INFO - Closed-form section-resultant check: max relative error = 9.944e-16
2026-09-04 00:33:38,485 - INFO - Coefficient ranges: d(X)=[0.746154, 1], r(X)=[1, 1.48485]
2026-09-04 00:33:38,486 - INFO - Max |z_E|/h = 0.05, max |z_rho|/h = 0.0544218; I1^rho is nonzero in general.
2026-09-04 00:33:38,545 - INFO - Compact-vs-expanded operator check: max abs=1.421e-14, relative=3.199e-16
2026-09-04 00:33:39,363 - INFO - S-S BVP candidate: guess=50 | lambda=68.534166545919 | interior_nodes=0 | mesh_nodes=602 | max_rms=9.996e-08
2026-09-04 00:33:39,426 - INFO - S-S BVP candidate: guess=100 | lambda=68.534166545919 | interior_nodes=0 | mesh_nodes=602 | max_rms=9.996e-08
202


FINAL S-S PINN RESULTS
Device          : cpu
Best epoch      : 50000
Best loss       : 1.280013220233e+04
Best lambda     : 68.868574511023
Omega [rad/s]   : 1.500892049770e+06
Frequency [Hz]  : 2.388743887682e+05
------------------------------------------------------------------------------
S-S endpoint diagnostics
X=0 | Phi       : 0.000000e+00
X=0 | Phi''     : 0.000000e+00
X=0 | Phi' free : 3.688256e+00
X=0 | Mcore     : 3.405597e+01
X=0 | Mbar      : 8.513994e-02
X=1 | Phi       : 0.000000e+00
X=1 | Phi''     : 0.000000e+00
X=1 | Phi' free : -4.073773e+00
X=1 | Mcore     : 4.170979e+01
X=1 | Mbar      : 1.042745e-01
Hard BC max abs : 0.000000e+00
Mcore RMS       : 3.807569e+01
Mbar RMS        : 9.518922e-02
------------------------------------------------------------------------------
Independent S-S BVP validation
Reference lambda : 68.534166542722
Lambda error [%] : 0.487943
Mass-weighted MAC: 0.9999059332
Mode rel. L2     : 9.698918e-03
Frequency error [%]: 0.243675
BVP interi